# Lecture 02 Advanced, v3: plain × 20 epochs, seeds [42, 7, 2026], validation loss, alias / made-up questions

v3 of `lora_glossary.ipynb`. Training data, LoRA settings, the original 35-question evaluation, glossary search and search + model are
the **same as v2** (copied unchanged into the run script). v3 adds three seeds, a validation split with per-epoch validation loss, the
separate v3 evaluation set (`alias`, `made_up`), `versions.json`, per-seed JSONs, a mean / min–max summary and `loss_curves_v3_plain_ep20.png`.

**This notebook runs the heavy work in the background** so a sleeping browser tab cannot freeze it:

| Cell | What | Time |
| --- | --- | --- |
| 1 GPU check, 2 install | as v2 | ~1–2 min |
| 3 SETUP | mounts Google Drive, sets `MyDrive/aie2026/lec02_v3/plain_ep20/` | <1 min |
| 4 SCRIPT | writes `/content/lec02_v3/run_plain_ep20.py` (all training and evaluation code) | instant |
| 5 LAUNCH | starts the script with `nohup … &`, returns after 20 s; refuses to start a second copy | 20 s |
| 6 POLL | status + last 20 log lines; rerun as often as you like | instant |
| 7 FINISH | when finished: summary, loss plot, zip (also copied to Drive) | ~1 min |

Expected background run time on a T4 ≈ 85–95 minutes. Progress bars are off (`disable_tqdm=True`, `report_to="none"`); the log gets a line every
5 steps, the training / validation losses, and one line per finished evaluation. If the runtime disconnects: reconnect, run cells 1–4, then 5
(finished parts — `shared/` files and `seed_<n>/done.json` — are skipped). QLoRA training is out of scope and is not done.

### Validation split and v3 evaluation set

**Validation split**: for every glossary term, the question `Define {term}.` with the glossary definition as the answer (58 records).
This template is not one of the two training templates or the alias template, and the cell checks that no validation question is an
evaluation question (original 35 or v3) and none is a training question. What it measures: whether the loss on a *new phrasing of known terms*
keeps falling or turns up while training loss goes to ~0 (overfitting to the two templates). It does not measure unknown terms.

**v3 evaluation set** (`eval_set_v3.json`, separate from the 35 so earlier numbers stay comparable):

| Group | n | Built from | Expected |
| --- | --- | --- | --- |
| `alias` | 12 | Acronyms and full names that the glossary itself lists (`HITL`, `Parameter-Efficient Fine-Tuning`, `SFT`, `PR`, ...) asked as `What is X?` | Definition of that term; `must_include` = 1-2 hand-picked distinctive words from the definition; **token F1 is the primary metric** for this group, keyword hit is secondary |
| `made_up` | 10 | Invented course-style names (`AIE_REVIEW_MODE`, `grade.md`, `Pi review buffer`, ...), checked to match no glossary term or alias; 5 contain a real glossary word as a search trap | "That term is not in the course glossary." |

Note for `augmented`: its alias training questions use `What does {alias} mean?`, so the alias *strings* were seen in training (the v3 questions themselves were not).

## 0. Check the Environment and Install Libraries

**What to Do**: Check that a GPU is allocated and install libraries pinned to verified versions.

**What to Look For**

- The first cell displays `Tesla T4`.
- The second cell ends with a list of versions (such as `transformers: 5.18.0`). Installation takes about 1 minute.

**Troubleshooting**

- "No GPU found": select **Runtime → Change runtime type → T4 GPU** and run the cell again. If demand prevents allocation, try again later.
- Red pip `ERROR: pip's dependency resolver ...` messages (about `cudf` or `pyarrow`) warn of conflicts with other packages preinstalled in Colab. They do not affect this exercise, so continue.

In [ ]:
# [Cell 1] GPU check
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU runtimes do not have nvidia-smi
    gpu_info = ""
print(gpu_info or "No GPU found. Select Runtime → Change runtime type → T4 GPU.")

In [ ]:
# [Cell 2] install
# Versions verified on a Colab T4 (Python 3.13, torch 2.11.0+cu130). Use the torch bundled with Colab
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 requires transformers 5.x
    "peft": "0.21.1",           # LoRA implementation
    "trl": "1.14.2",            # Training loop (SFTTrainer)
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # The bundled torchao (0.10) is incompatible with peft 0.21 and causes ImportError during training. Remove it since it is unused
%pip install -q {spec} "bitsandbytes==0.50.*"   # v3: bitsandbytes installed up front only so its version is recorded (no 4-bit training)

# Dependency warnings about cudf-cu13 and pyarrow appear as ERROR, but can be ignored because cudf is unused
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## Setup and launch

Run cells 3, 4 and 5 once. Cell 5 returns after about 20 seconds; training continues in the background.

In [ ]:
# [Cell 3] SETUP: mount Google Drive and set paths (run again after any reconnect)
import os, sys, json, time, shlex, shutil, subprocess
from pathlib import Path
RUN_NAME = "plain_ep20"
USE_DRIVE = True   # strongly recommended: results and the log survive a disconnect
LOCAL = Path("/content/lec02_v3"); LOCAL.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT = Path("/content/drive/MyDrive/aie2026/lec02_v3")
if USE_DRIVE:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        try:
            drive.mount("/content/drive")   # school account: allow the pop-up, pick the school Google account, allow all permissions
        except Exception as e:
            print(f"Drive mount failed ({type(e).__name__}: {e}).\n"
                  "Try: allow pop-ups / third-party cookies for colab.research.google.com, pick the SAME school account that owns "
                  "this notebook, tick every permission, then rerun this cell. If it still fails: Runtime -> Disconnect and delete "
                  "runtime, reconnect (T4) and rerun. As a last resort set USE_DRIVE = False (results then live only in /content).")
            raise
    OUT = DRIVE_ROOT / RUN_NAME
else:
    OUT = LOCAL / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
RUN_PY = LOCAL / "run_plain_ep20.py"
LOG = OUT / "log.txt"
LAUNCH_WAIT_S = 20

def run_state():
    """Lock contents if the detached run is alive on this VM, else None"""
    lock = OUT / "run.lock"
    try: d = json.loads(lock.read_text())
    except (OSError, ValueError): return None
    if d.get("boot_id") != Path("/proc/sys/kernel/random/boot_id").read_text().strip(): return None
    try: os.kill(int(d["pid"]), 0)
    except (OSError, ValueError, KeyError): return None
    return d

def tail(n=20):
    if not LOG.exists(): return "(no log yet)"
    return "\n".join(LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-n:])

print("Outputs ->", OUT, "| script ->", RUN_PY, "| running now:", bool(run_state()))

In [ ]:
%%writefile /content/lec02_v3/run_plain_ep20.py
# [Cell 4] SCRIPT: written to disk, not run here; the LAUNCH cell runs it in the background
"""Lecture 02 Advanced v3, plain x 20 epochs, seeds [42, 7, 2026]. Written by the notebook; launched detached.
Usage: python -u run_plain_ep20.py [--stage all|shared|seed --seed N|summary]"""

# ---------------------------------------------------------------- runtime: log, lock, child stages
import os, sys, json, time, socket, signal, subprocess, argparse, contextlib
from pathlib import Path
for _k in ("TQDM_DISABLE", "HF_HUB_DISABLE_PROGRESS_BARS", "HF_DATASETS_DISABLE_PROGRESS_BARS"):
    os.environ.setdefault(_k, "1")            # no progress bars: plain log lines only
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
DRY_RUN = os.environ.get("LEC02_DRY_RUN") == "1"   # CPU test mode used only by validate.py: stub model, no training

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

def display(x):  # notebook display() -> plain text in the log
    print(x.to_string() if hasattr(x, "to_string") else x, flush=True)

def boot_id():
    try: return Path("/proc/sys/kernel/random/boot_id").read_text().strip()
    except OSError: return socket.gethostname()

def lock_owner(lock):
    """Lock contents if the process that wrote it is still alive on this VM, else None"""
    try: d = json.loads(lock.read_text())
    except (OSError, ValueError): return None
    if d.get("boot_id") != boot_id(): return None
    try: os.kill(int(d["pid"]), 0)
    except (OSError, ValueError, KeyError): return None
    try:
        if d.get("script") and d["script"] not in Path(f"/proc/{d['pid']}/cmdline").read_bytes().decode(errors="replace"):
            return None                       # pid reused by another program
    except OSError: pass
    return d

def acquire_lock(lock):
    other = lock_owner(lock)
    if other and int(other["pid"]) != os.getpid():
        log(f"another run is active (pid {other['pid']}, started {other.get('started')}); this copy exits")
        sys.exit(3)
    lock.write_text(json.dumps({"pid": os.getpid(), "boot_id": boot_id(), "script": Path(__file__).name,
                                "started": time.strftime("%Y-%m-%d %H:%M:%S")}))
    time.sleep(1)
    if json.loads(lock.read_text()).get("pid") != os.getpid():   # lost a simultaneous start
        log("another run took the lock; this copy exits"); sys.exit(3)
    def _stop(signum, frame):
        log(f"received signal {signum}; stopping"); release_lock(lock); sys.exit(128 + signum)
    signal.signal(signal.SIGTERM, _stop)

def release_lock(lock):
    try:
        if json.loads(lock.read_text()).get("pid") == os.getpid(): lock.unlink()
    except (OSError, ValueError): pass

def run_stage(args, retries=1):
    """Run one stage of this script in a fresh child process (fresh GPU memory). Output goes to the same log"""
    for attempt in range(1, retries + 2):
        log(f"==> stage {' '.join(args)} (attempt {attempt})")
        rc = subprocess.call([sys.executable, "-u", __file__, *args])
        log(f"<== stage {' '.join(args)} exit code {rc}")
        if rc == 0: return True
    return False

def require_gpu():
    if DRY_RUN: return
    import torch
    if not torch.cuda.is_available():
        log("ERROR: no GPU. Runtime -> Change runtime type -> T4 GPU, then run the SETUP and LAUNCH cells again"); sys.exit(2)
    log("GPU:", torch.cuda.get_device_name(0))

def heartbeat_callback():
    """One log line every 5 optimizer steps (training loss / validation loss lines come from the Trainer itself)"""
    from transformers import TrainerCallback
    class Heartbeat(TrainerCallback):
        def on_train_begin(self, args, state, control, **kw):
            self.t0 = time.time(); log(f"training starts: {state.max_steps} steps")
        def on_step_end(self, args, state, control, **kw):
            if state.global_step % 5 == 0 or state.global_step == state.max_steps:
                log(f"step {state.global_step}/{state.max_steps} | epoch {state.epoch:.2f} | {time.time() - self.t0:.0f} s")
    return Heartbeat()

# ---------------------------------------------------------------- 1. settings (v3)
VARIANT = "plain"
EPOCHS = 20
SEEDS = [42, 7, 2026]   # 42 = the v2 run's seed; two extra seeds for spread
assert VARIANT in ("plain", "augmented") and isinstance(EPOCHS, int) and EPOCHS >= 1
RUN_NAME = f"{VARIANT}_ep{EPOCHS}"
CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "variant": VARIANT,
    "run_name": RUN_NAME,
    "seeds": SEEDS,
    "num_train_epochs": EPOCHS,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 150,
    "gen": {"do_sample": False},
    "work_dir": "/content/qlora_lab",
    "drive_dir": "/content/drive/MyDrive/aie2026/lec02_v3",
}
WORK = Path(os.environ.get("LEC02_WORK", CONFIG["work_dir"])); WORK.mkdir(parents=True, exist_ok=True)   # local, scratch only
OUT = Path(os.environ.get("LEC02_OUT") or Path(CONFIG["drive_dir"]) / RUN_NAME)                          # results (Drive)
if not OUT.parent.exists():
    sys.exit(f"Output folder {OUT.parent} does not exist: run the SETUP cell (it mounts Google Drive) first")
OUT.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- 2. embedded data (v2 cell, unchanged)
# Embedded data (generated from the course repository; do not edit by hand)
GLOSSARY = {'version': '2026-draft-1',
 'note': 'Draft glossary for Lecture 01–03 and Lecture 05 of AIE2026. Replace content once the lecture slides are finalized.',
 'terms': [{'id': 'ai-engineering',
            'term': 'AI engineering',
            'aliases': ['AI Engineering'],
            'lecture': 1,
            'category': 'Overview',
            'definition': 'The technical field of designing, building, evaluating, and operating useful applications and systems using '
                          'pretrained foundation models (mainly LLMs) as components. It focuses on selecting models, connecting them to '
                          'data and tools, and iterating on evaluation and improvement, rather than building models from scratch.',
            'related': ['llm', 'evaluation-loop']},
           {'id': 'evaluation-loop',
            'term': 'Evaluation and improvement cycle',
            'aliases': ['improvement cycle'],
            'lecture': 1,
            'category': 'Overview',
            'definition': 'A development approach that repeats build → measure → improve. Since AI application outputs vary '
                          'probabilistically, judge quality using evaluation data and metrics rather than intuition, then improve prompts, '
                          'data, models, and tools based on the results. In this course, we repeat this cycle across lectures using a '
                          'learning support agent.',
            'related': ['ai-engineering', 'hallucination']},
           {'id': 'llm',
            'term': 'LLM (Large Language Model)',
            'aliases': ['large language model'],
            'lecture': 1,
            'category': 'LLM basics',
            'definition': 'A model trained on large amounts of text that generates text by predicting the tokens following its input. It '
                          'can handle many tasks, including question answering, summarization, and code generation, simply by changing the '
                          'instructions (prompt).',
            'related': ['token', 'prompt', 'pretraining']},
           {'id': 'token',
            'term': 'Token',
            'aliases': ['token'],
            'lecture': 1,
            'category': 'LLM basics',
            'definition': 'The smallest unit an LLM uses to process text. Text is split into fragments of words or characters. API '
                          'charges, rate limits, and context window sizes are often counted in tokens. Tokenization varies by language; '
                          'for example, Japanese tends to require more tokens per character than English.',
            'related': ['context-window', 'rate-limit']},
           {'id': 'context-window',
            'term': 'Context window',
            'aliases': ['context length', 'context window'],
            'lecture': 1,
            'category': 'LLM basics',
            'definition': 'The maximum total number of input and output tokens an LLM can handle in one call. Conversation history and '
                          'tool results must also fit within this limit, making the choice of what to pass to the model important.',
            'related': ['token', 'agent-loop']},
           {'id': 'prompt',
            'term': 'Prompt',
            'aliases': ['prompt'],
            'lecture': 1,
            'category': 'LLM basics',
            'definition': 'Input text passed to an LLM. In addition to questions, it can include roles, constraints, output formats, and '
                          'examples to guide the output.',
            'related': ['system-prompt', 'llm']},
           {'id': 'system-prompt',
            'term': 'System prompt',
            'aliases': ['instruction text', 'instructions', 'system instruction'],
            'lecture': 1,
            'category': 'LLM basics',
            'definition': "Input that specifies model behavior (role, tone, rules, and so on) separately from the user's question. In "
                          'Pydantic AI, it is provided through the Agent instructions argument.',
            'related': ['prompt', 'guardrail']},
           {'id': 'temperature',
            'term': 'temperature',
            'aliases': ['temperature parameter'],
            'lecture': 1,
            'category': 'LLM basics',
            'definition': 'A setting that adjusts randomness when choosing the next token. Lower values make outputs more stable; higher '
                          'values make them more varied. This is one reason the same question can yield different answers.',
            'related': ['llm', 'evaluation-loop']},
           {'id': 'hallucination',
            'term': 'Hallucination',
            'aliases': ['fabrication', 'hallucination'],
            'lecture': 1,
            'category': 'LLM basics',
            'definition': 'The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more '
                          'likely when asked about things the model does not know, such as recent information or organization- or '
                          'course-specific details. Mitigations include consulting external data through tools, asking for evidence, and '
                          'detecting errors through evaluation.',
            'related': ['tool', 'evaluation-loop']},
           {'id': 'api',
            'term': 'API',
            'aliases': ['Application Programming Interface', 'Web API'],
            'lecture': 1,
            'category': 'Development basics',
            'definition': 'An interface through which a program calls the functionality of another service. In this course, we use an LLM '
                          'by calling the Gemini API from Python.',
            'related': ['api-key', 'rate-limit']},
           {'id': 'api-key',
            'term': 'API key',
            'aliases': ['API key'],
            'lecture': 1,
            'category': 'Development basics',
            'definition': 'A secret string that identifies the person calling an API. If it leaks, someone else can consume your usage '
                          'allowance, so do not put it directly in code, commit it, or paste it into chat. In this course, register it as '
                          'a Codespaces secret and read it through the GEMINI_API_KEY environment variable.',
            'related': ['codespaces-secret', 'environment-variable']},
           {'id': 'environment-variable',
            'term': 'Environment variable',
            'aliases': ['environment variable', 'env'],
            'lecture': 1,
            'category': 'Development basics',
            'definition': 'A value set outside a program (in the OS or execution environment) that the program can read by name. Used to '
                          'separate secrets such as API keys and environment-specific settings from code.',
            'related': ['api-key', 'codespaces-secret']},
           {'id': 'rate-limit',
            'term': 'Rate limit',
            'aliases': ['rate limit', '429 error'],
            'lecture': 1,
            'category': 'Development basics',
            'definition': 'A limit on requests or tokens within a given time period. Exceeding it returns errors such as HTTP 429. Free '
                          'tiers have particularly strict limits, and agents reach them easily because they call the LLM multiple times '
                          'for one question.',
            'related': ['api', 'agent-loop', 'default-model']},
           {'id': 'codespaces',
            'term': 'GitHub Codespaces',
            'aliases': ['Codespaces', 'Codespace'],
            'lecture': 1,
            'category': 'Development environment',
            'definition': 'A cloud development environment provided by GitHub. Open VS Code in a browser and immediately use an '
                          'environment configured by the repository. Personal accounts have a monthly free allowance; if you do not stop '
                          'the Codespace when idle, it continues consuming that allowance.',
            'related': ['devcontainer', 'codespaces-secret']},
           {'id': 'devcontainer',
            'term': 'devcontainer',
            'aliases': ['Dev Container', 'devcontainer.json'],
            'lecture': 1,
            'category': 'Development environment',
            'definition': 'A mechanism for defining a development environment as a container. Specify the Python version, extensions, and '
                          'startup commands in .devcontainer/devcontainer.json so everyone opening Codespaces gets the same environment.',
            'related': ['codespaces']},
           {'id': 'codespaces-secret',
            'term': 'Codespaces secret',
            'aliases': ['Codespaces secrets'],
            'lecture': 1,
            'category': 'Development environment',
            'definition': 'A secret value registered in your GitHub account settings. It is passed as an environment variable when you '
                          'start Codespaces for a selected repository. When registering it, select the repository you will use (your own '
                          'fork). If registered while a Codespace is running, restart the Codespace.',
            'related': ['api-key', 'environment-variable', 'codespaces']},
           {'id': 'fork',
            'term': 'fork',
            'aliases': ['forking'],
            'lecture': 1,
            'category': 'Git/GitHub',
            'definition': "Copying someone else's entire repository to your own account. In this course, fork the course repository and "
                          'work and submit within your own fork.',
            'related': ['branch', 'pull-request']},
           {'id': 'branch',
            'term': 'Branch',
            'aliases': ['branch'],
            'lecture': 1,
            'category': 'Git/GitHub',
            'definition': 'A mechanism for separating streams of work. Try changes without breaking main, then merge them into main '
                          'through a PR once ready. In this course, create a working branch such as lec01 for each lecture.',
            'related': ['pull-request', 'fork']},
           {'id': 'pull-request',
            'term': 'Pull Request (PR)',
            'aliases': ['PR', 'Pull Request'],
            'lecture': 1,
            'category': 'Git/GitHub',
            'definition': 'A mechanism for requesting that changes from one branch be merged into another. It supports reviewing '
                          'differences, comments, and displaying automated test results. Submit work for this course as a PR from your '
                          'working branch to main within your own fork.',
            'related': ['branch', 'answer-md']},
           {'id': 'github-copilot',
            'term': 'GitHub Copilot',
            'aliases': ['Copilot'],
            'lecture': 1,
            'category': 'AI coding',
            'definition': "GitHub's AI coding assistance. It supports code completion in the editor, questions in chat, and editing "
                          'multiple files in Agent mode. Copilot Free limits completion counts and chat usage.',
            'related': ['code-completion', 'coding-agent']},
           {'id': 'code-completion',
            'term': 'Code completion',
            'aliases': ['inline completion', 'inline suggestion'],
            'lecture': 1,
            'category': 'AI coding',
            'definition': 'A feature in which AI suggests code to follow partially written code or comments. Accept suggestions with Tab '
                          'or a similar action. Suggestions may be incorrect, so read and understand them before accepting.',
            'related': ['github-copilot']},
           {'id': 'coding-agent',
            'term': 'Coding agent',
            'aliases': ['Coding Agent', 'AI coding agent'],
            'lecture': 1,
            'category': 'AI coding',
            'definition': 'AI that autonomously carries out requested programming work over multiple steps, using tools to read and write '
                          'files and run commands. Examples include Claude Code, Copilot Agent mode, and Pi. Internally, it has the same '
                          'LLM + loop + tools structure as an AI agent.',
            'related': ['ai-agent', 'harness']},
           {'id': 'pretraining',
            'term': 'Pretraining',
            'aliases': ['pre-training', 'pretraining'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'The initial stage of training a model to predict the next token using large amounts of general-purpose text. '
                          'Much of its language knowledge and general common sense is acquired here.',
            'related': ['fine-tuning', 'llm']},
           {'id': 'fine-tuning',
            'term': 'Fine-tuning',
            'aliases': ['fine-tuning', 'additional training', 'FT'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'Further training a pretrained model on data suited to a specific task or format. It is suitable for teaching '
                          'output format, tone, or behavior in a specific domain. For up-to-date facts, consulting external data through '
                          'tools or RAG is often more suitable.',
            'related': ['sft', 'peft', 'lora']},
           {'id': 'sft',
            'term': 'Supervised Fine-Tuning (SFT)',
            'aliases': ['SFT', 'Supervised Fine-Tuning', 'instruction tuning'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A method of further training a model using pairs of inputs and desired outputs, such as questions and model '
                          'answers. A representative method for teaching instruction-following behavior.',
            'related': ['fine-tuning', 'lora']},
           {'id': 'peft',
            'term': 'PEFT',
            'aliases': ['Parameter-Efficient Fine-Tuning'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'An umbrella term for methods that train only a small subset of model parameters or a small set of added '
                          'parameters instead of all parameters. This can greatly reduce GPU memory and storage requirements. LoRA is a '
                          'representative example.',
            'related': ['lora', 'fine-tuning']},
           {'id': 'lora',
            'term': 'LoRA',
            'aliases': ['Low-Rank Adaptation'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices '
                          '(low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results '
                          '(adapters) in small files. The Lecture 02 exercises cover LoRA.',
            'related': ['peft', 'lora-rank', 'qlora']},
           {'id': 'lora-rank',
            'term': 'Rank (LoRA r)',
            'aliases': ['rank', 'r'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A hyperparameter that determines the size of the low-rank matrices used in LoRA. A larger value increases '
                          'expressive capacity, but also increases the number of trainable parameters and memory use.',
            'related': ['lora']},
           {'id': 'qlora',
            'term': 'QLoRA',
            'aliases': ['Quantized LoRA'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A method that saves memory by quantizing base-model weights to, for example, 4 bits, then trains with LoRA. '
                          'This lets smaller GPUs handle larger models. In the Lecture 02 exercises, the model fits in GPU memory, so we '
                          'train with LoRA and compare memory and output differences when loading in 4-bit in the advanced section.',
            'related': ['lora', 'quantization']},
           {'id': 'quantization',
            'term': 'Quantization',
            'aliases': ['quantization'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A technique for representing model weights with fewer bits (for example, from 16 bits to 4 bits) to reduce '
                          'memory use and computation. Accuracy may decrease slightly.',
            'related': ['qlora']},
           {'id': 'learning-rate',
            'term': 'Learning rate',
            'aliases': ['learning rate', 'lr'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A hyperparameter that determines how far parameters move in one update. Too large a value makes training '
                          'unstable; too small a value slows progress.',
            'related': ['epoch', 'overfitting']},
           {'id': 'epoch',
            'term': 'Epoch',
            'aliases': ['epoch'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'One pass through the entire training dataset is one epoch. Too many epochs can cause overfitting.',
            'related': ['overfitting', 'learning-rate']},
           {'id': 'overfitting',
            'term': 'Overfitting',
            'aliases': ['overfitting'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A state in which the model fits training data excessively and performs worse on unseen inputs. Detect it by '
                          'measuring performance on validation data not used for training.',
            'related': ['epoch', 'evaluation-loop']},
           {'id': 'huggingface-hub',
            'term': 'Hugging Face Hub',
            'aliases': ['Hugging Face', 'HF Hub'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A platform for sharing and downloading public models and datasets. Used to obtain open-weight models and '
                          'publish LoRA adapters.',
            'related': ['lora', 'pretraining']},
           {'id': 'ai-agent',
            'term': 'AI agent',
            'aliases': ['Agent'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'A system that achieves a goal through a loop in which an LLM decides to call tools, examines the results, and '
                          'decides its next action. Its basic structure is "model + loop + tools".',
            'related': ['agent-loop', 'tool', 'harness']},
           {'id': 'agent-loop',
            'term': 'Agent loop',
            'aliases': ['agent loop', 'tool-calling loop'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'The process of passing input to the model → executing any tool calls returned by the model → returning results '
                          'to the model, repeated until the model gives a final answer. Multiple LLM calls for one question have a '
                          'substantial effect on time, charges, and rate limits.',
            'related': ['ai-agent', 'tool', 'rate-limit']},
           {'id': 'tool',
            'term': 'Tool (Tool use / Function calling)',
            'aliases': ['tool call', 'Function calling', 'Tool use', 'tool'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'A function made available for the model to call. The model looks at its name, description, and argument types, '
                          'chooses arguments when needed, and requests a call; the program executes it. Used for search, calculations, '
                          'file operations, external API calls, and more.',
            'related': ['tool-description', 'agent-loop']},
           {'id': 'tool-description',
            'term': 'Tool description',
            'aliases': ['docstring', 'tool specification', 'tool schema'],
            'lecture': 3,
            'category': 'Agents',
            'definition': "Text that explains a tool's purpose and argument meanings to the model. In Pydantic AI, it is generated "
                          'automatically from the function docstring and type hints. Since the model reads it to decide whether and how to '
                          'call a tool, its wording can greatly affect behavior.',
            'related': ['tool', 'pydantic-ai']},
           {'id': 'structured-output',
            'term': 'Structured output',
            'aliases': ['structured output', 'JSON output'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'A mechanism for receiving model outputs according to specified types or a JSON schema. This makes programmatic '
                          'postprocessing easier. In Pydantic AI, specify a Pydantic model in output_type.',
            'related': ['pydantic-ai']},
           {'id': 'harness',
            'term': 'Harness',
            'aliases': ['agent harness', 'agent wrapper'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'A program that surrounds an LLM with a loop, tools, a user interface (UI), and so on to enable actual work. '
                          'Claude Code and Pi are examples. Pi is known as a thin harness with just four built-in tools: read, write, '
                          'edit, and bash.',
            'related': ['coding-agent', 'pi']},
           {'id': 'pi',
            'term': 'Pi',
            'aliases': ['Pi coding agent', 'pi agent'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'An open-source (MIT-licensed) coding agent that runs in the terminal. It has minimal features and a minimal '
                          'system prompt, designed for users to extend. Version 1.0 was released in October 2026. Its npm package name is '
                          '@earendil-works/pi-coding-agent.',
            'related': ['harness', 'coding-agent']},
           {'id': 'pydantic-ai',
            'term': 'Pydantic AI',
            'aliases': ['pydantic-ai', 'PydanticAI'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'A framework for building agents in Python. It uses Pydantic types to define tool arguments and outputs and '
                          'handles many models, such as Gemini and OpenAI, with the same code style. Test models let you check behavior '
                          'without API calls. It is the main framework for Lecture 03.',
            'related': ['test-model', 'structured-output', 'adk']},
           {'id': 'adk',
            'term': 'ADK (Agent Development Kit)',
            'aliases': ['ADK', 'Agent Development Kit', 'google-adk'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'An open-source agent development framework released by Google. It integrates closely with Gemini, and adk web '
                          'lets you visually follow tool calls. Version 2.0 introduces breaking changes from the 1.x series. This course '
                          'covers it in the Advanced homework.',
            'related': ['pydantic-ai', 'ai-agent']},
           {'id': 'mcp',
            'term': 'MCP (Model Context Protocol)',
            'aliases': ['MCP', 'Model Context Protocol'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'An open standard protocol connecting AI applications to external tools and data sources. Tools exposed through '
                          'an MCP server can be used in a common way from compatible agents and editors (MCP clients). In Lecture 05, you '
                          'will experience this by turning glossary search into an MCP server.',
            'related': ['tool', 'ai-agent', 'mcp-server', 'mcp-client']},
           {'id': 'guardrail',
            'term': 'Guardrail',
            'aliases': ['guardrail'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'A constraint or check designed to prevent undesirable agent behavior or outputs. Methods include rules in '
                          'instructions, restricting tool permissions, checking outputs, and requesting human confirmation before '
                          'important operations.',
            'related': ['human-in-the-loop', 'system-prompt']},
           {'id': 'human-in-the-loop',
            'term': 'Human-in-the-loop',
            'aliases': ['HITL', 'human approval', 'confirmation step'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'Incorporating human confirmation or approval into agent processing. Used before operations that are difficult '
                          'to undo, such as writing or sending data. In Pydantic AI, tools with requires_approval=True support this.',
            'related': ['guardrail', 'tool']},
           {'id': 'test-model',
            'term': 'TestModel',
            'aliases': ['test model'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'A fake model included in Pydantic AI for testing. It calls tools and returns dummy answers without calling a '
                          'real LLM. It lets you test agent wiring (whether tools are registered, called, and return results) without an '
                          'API key or usage allowance.',
            'related': ['pydantic-ai', 'rate-limit']},
           {'id': 'homework-levels',
            'term': '3 homework levels',
            'aliases': ['3 levels', 'homework levels'],
            'lecture': 1,
            'category': 'Course',
            'definition': 'The structure of AIE2026 homework. There are three levels: "Basic (review lecture content)", "Implementation '
                          '(write your own code)", and "Advanced (research and experimentation)". Completing any one level completes the '
                          'homework for that lecture. As a rule, it is not graded.',
            'related': ['answer-md']},
           {'id': 'answer-md',
            'term': 'answer.md',
            'aliases': ['answer file'],
            'lecture': 1,
            'category': 'Course',
            'definition': 'The file for recording exercise and homework results for each AIE2026 lecture. Located at '
                          'lectures/NN/answer.md. Record the level attempted, what you did, results, and observations, then submit by '
                          'opening a PR from your working branch to main in your own fork.',
            'related': ['pull-request', 'homework-levels']},
           {'id': 'lec-start-tag',
            'term': 'lecNN-start tag',
            'aliases': ['start tag', 'lec03-start'],
            'lecture': 1,
            'category': 'Course',
            'definition': "A Git tag on the starting code for each lecture's exercises in the AIE2026 course repository (for example, "
                          'lec03-start). Even if you have not finished the previous lecture, creating a branch from this tag lets you '
                          "start the current lecture's exercises.",
            'related': ['branch']},
           {'id': 'default-model',
            'term': 'Default model',
            'aliases': ['default AI model', 'gemini-3.5-flash-lite'],
            'lecture': 1,
            'category': 'Course',
            'definition': 'The standard model for AIE2026 exercises and homework is gemini-3.5-flash-lite, chosen with free-tier rate '
                          'limits and stable course operation in mind. You may switch models with the AIE_MODEL environment variable or '
                          'use your own API subscription.',
            'related': ['rate-limit', 'api-key']},
           {'id': 'external-tool',
            'term': 'External tool',
            'aliases': ['external tool', 'external service'],
            'lecture': 5,
            'category': 'MCP',
            'definition': 'Using capabilities outside the agent program, such as web retrieval, file operations, databases, and '
                          'third-party APIs, as tools. Lecture 03 function tools run within the same process, whereas external tools are '
                          'called across processes or machines. MCP is a common protocol for this.',
            'related': ['tool', 'mcp', 'mcp-server']},
           {'id': 'mcp-server',
            'term': 'MCP server',
            'aliases': ['MCP server'],
            'lecture': 5,
            'category': 'MCP',
            'definition': 'The program that exposes tools (and resources or prompts) through MCP. It exposes tool names, descriptions, and '
                          'argument schemas, executes tools in response to client requests, and returns results. In this course, glossary '
                          'search becomes an MCP server (app/mcp_server.py). Many ready-made servers are available, including fetch, '
                          'filesystem, and GitHub.',
            'related': ['mcp', 'mcp-client', 'mcp-transport', 'fetch-server']},
           {'id': 'mcp-client',
            'term': 'MCP client',
            'aliases': ['MCP client', 'MCP host'],
            'lecture': 5,
            'category': 'MCP',
            'definition': 'The component that connects to MCP servers, passes their tools to the model, and calls tools as directed by the '
                          "model. Agent programs (Pydantic AI's MCPToolset) and editors or applications such as Copilot Chat, Claude "
                          'Desktop, and Cursor can serve as clients. A benefit of MCP is using the same server from different clients.',
            'related': ['mcp', 'mcp-server', 'mcp-config']},
           {'id': 'mcp-transport',
            'term': 'Transport (stdio / Streamable HTTP)',
            'aliases': ['stdio', 'Streamable HTTP', 'transport', 'MCP transport'],
            'lecture': 5,
            'category': 'MCP',
            'definition': 'The communication channel between an MCP client and server. With stdio, the client starts the server as a '
                          'subprocess and exchanges messages through standard input/output (simple and used on the same machine). With '
                          'Streamable HTTP, the server listens at a URL, allowing connections from other processes or machines (access '
                          'scope and authentication must be considered).',
            'related': ['mcp-server', 'mcp-client']},
           {'id': 'mcp-config',
            'term': 'mcpServers configuration',
            'aliases': ['mcpServers', 'MCP configuration file', 'mcp_servers.json', 'mcp.json'],
            'lecture': 5,
            'category': 'MCP',
            'definition': 'A configuration file listing MCP servers to use. For each server, specify a name and a startup command (stdio) '
                          'or destination URL (HTTP). Claude Desktop, Cursor, and Pydantic AI read the same mcpServers format (VS Code / '
                          'Copilot Chat uses servers). This course uses app/mcp_servers.json. Adding tools with just a few configuration '
                          'lines is central to the MCP experience.',
            'related': ['mcp-client', 'tool-namespace']},
           {'id': 'tool-namespace',
            'term': 'Tool namespace',
            'aliases': ['tool-name prefix', 'prefix', 'name collision'],
            'lecture': 5,
            'category': 'MCP',
            'definition': 'Adding a prefix such as "server_name_tool_name" to distinguish identically named tools when using multiple MCP '
                          "servers. This course's app/agent.py automatically adds the server name from the configuration (for example, "
                          "glossary_search_glossary; Pydantic AI's load_mcp_toolsets does the same). More tools mean more descriptions "
                          'sent to the model, affecting tool-selection accuracy and token counts.',
            'related': ['mcp-config', 'tool-description', 'rate-limit']},
           {'id': 'fetch-server',
            'term': 'fetch server',
            'aliases': ['mcp-server-fetch', 'fetch', 'web retrieval tool'],
            'lecture': 5,
            'category': 'MCP',
            'definition': 'A ready-made MCP server that retrieves a specified URL and returns the page body as Markdown (written in '
                          'Python; started with uvx mcp-server-fetch). This course uses it as an example of "adding an external tool just '
                          'through configuration". It actually accesses URLs chosen by the model, so users must understand where it sends '
                          'data and what it sends.',
            'related': ['mcp-server', 'external-tool', 'guardrail']}]}

SFT_SHA256 = 'df11fef8e6251754ff7c814ed2f7d979b96bd927a252aef36819d1ddf361386f'  # sha256 of glossary_sft.jsonl produced by scripts/make_sft_dataset.py
SFT_JSONL_LINES = [
    '{"messages": [{"role": "user", "content": "What is AI engineering?"}, {"role": "assistant", "content": "The technical field of designing, building, evaluating, and operating useful applications and systems using pretrained foundation models (mainly LLMs) as components. It focuses on selecting models, connecting them to data and tools, and iterating on evaluation and improvement, rather than building models from scratch."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain AI engineering."}, {"role": "assistant", "content": "The technical field of designing, building, evaluating, and operating useful applications and systems using pretrained foundation models (mainly LLMs) as components. It focuses on selecting models, connecting them to data and tools, and iterating on evaluation and improvement, rather than building models from scratch."}]}',
    '{"messages": [{"role": "user", "content": "What is Evaluation and improvement cycle?"}, {"role": "assistant", "content": "A development approach that repeats build → measure → improve. Since AI application outputs vary probabilistically, judge quality using evaluation data and metrics rather than intuition, then improve prompts, data, models, and tools based on the results. In this course, we repeat this cycle across lectures using a learning support agent."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Evaluation and improvement cycle."}, {"role": "assistant", "content": "A development approach that repeats build → measure → improve. Since AI application outputs vary probabilistically, judge quality using evaluation data and metrics rather than intuition, then improve prompts, data, models, and tools based on the results. In this course, we repeat this cycle across lectures using a learning support agent."}]}',
    '{"messages": [{"role": "user", "content": "What is LLM (Large Language Model)?"}, {"role": "assistant", "content": "A model trained on large amounts of text that generates text by predicting the tokens following its input. It can handle many tasks, including question answering, summarization, and code generation, simply by changing the instructions (prompt)."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain LLM (Large Language Model)."}, {"role": "assistant", "content": "A model trained on large amounts of text that generates text by predicting the tokens following its input. It can handle many tasks, including question answering, summarization, and code generation, simply by changing the instructions (prompt)."}]}',
    '{"messages": [{"role": "user", "content": "What is Token?"}, {"role": "assistant", "content": "The smallest unit an LLM uses to process text. Text is split into fragments of words or characters. API charges, rate limits, and context window sizes are often counted in tokens. Tokenization varies by language; for example, Japanese tends to require more tokens per character than English."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Token."}, {"role": "assistant", "content": "The smallest unit an LLM uses to process text. Text is split into fragments of words or characters. API charges, rate limits, and context window sizes are often counted in tokens. Tokenization varies by language; for example, Japanese tends to require more tokens per character than English."}]}',
    '{"messages": [{"role": "user", "content": "What is Context window?"}, {"role": "assistant", "content": "The maximum total number of input and output tokens an LLM can handle in one call. Conversation history and tool results must also fit within this limit, making the choice of what to pass to the model important."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Context window."}, {"role": "assistant", "content": "The maximum total number of input and output tokens an LLM can handle in one call. Conversation history and tool results must also fit within this limit, making the choice of what to pass to the model important."}]}',
    '{"messages": [{"role": "user", "content": "What is Prompt?"}, {"role": "assistant", "content": "Input text passed to an LLM. In addition to questions, it can include roles, constraints, output formats, and examples to guide the output."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Prompt."}, {"role": "assistant", "content": "Input text passed to an LLM. In addition to questions, it can include roles, constraints, output formats, and examples to guide the output."}]}',
    '{"messages": [{"role": "user", "content": "What is System prompt?"}, {"role": "assistant", "content": "Input that specifies model behavior (role, tone, rules, and so on) separately from the user\'s question. In Pydantic AI, it is provided through the Agent instructions argument."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain System prompt."}, {"role": "assistant", "content": "Input that specifies model behavior (role, tone, rules, and so on) separately from the user\'s question. In Pydantic AI, it is provided through the Agent instructions argument."}]}',
    '{"messages": [{"role": "user", "content": "What is temperature?"}, {"role": "assistant", "content": "A setting that adjusts randomness when choosing the next token. Lower values make outputs more stable; higher values make them more varied. This is one reason the same question can yield different answers."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain temperature."}, {"role": "assistant", "content": "A setting that adjusts randomness when choosing the next token. Lower values make outputs more stable; higher values make them more varied. This is one reason the same question can yield different answers."}]}',
    '{"messages": [{"role": "user", "content": "What is Hallucination?"}, {"role": "assistant", "content": "The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely when asked about things the model does not know, such as recent information or organization- or course-specific details. Mitigations include consulting external data through tools, asking for evidence, and detecting errors through evaluation."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Hallucination."}, {"role": "assistant", "content": "The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely when asked about things the model does not know, such as recent information or organization- or course-specific details. Mitigations include consulting external data through tools, asking for evidence, and detecting errors through evaluation."}]}',
    '{"messages": [{"role": "user", "content": "What is API?"}, {"role": "assistant", "content": "An interface through which a program calls the functionality of another service. In this course, we use an LLM by calling the Gemini API from Python."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain API."}, {"role": "assistant", "content": "An interface through which a program calls the functionality of another service. In this course, we use an LLM by calling the Gemini API from Python."}]}',
    '{"messages": [{"role": "user", "content": "What is API key?"}, {"role": "assistant", "content": "A secret string that identifies the person calling an API. If it leaks, someone else can consume your usage allowance, so do not put it directly in code, commit it, or paste it into chat. In this course, register it as a Codespaces secret and read it through the GEMINI_API_KEY environment variable."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain API key."}, {"role": "assistant", "content": "A secret string that identifies the person calling an API. If it leaks, someone else can consume your usage allowance, so do not put it directly in code, commit it, or paste it into chat. In this course, register it as a Codespaces secret and read it through the GEMINI_API_KEY environment variable."}]}',
    '{"messages": [{"role": "user", "content": "What is Environment variable?"}, {"role": "assistant", "content": "A value set outside a program (in the OS or execution environment) that the program can read by name. Used to separate secrets such as API keys and environment-specific settings from code."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Environment variable."}, {"role": "assistant", "content": "A value set outside a program (in the OS or execution environment) that the program can read by name. Used to separate secrets such as API keys and environment-specific settings from code."}]}',
    '{"messages": [{"role": "user", "content": "What is Rate limit?"}, {"role": "assistant", "content": "A limit on requests or tokens within a given time period. Exceeding it returns errors such as HTTP 429. Free tiers have particularly strict limits, and agents reach them easily because they call the LLM multiple times for one question."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Rate limit."}, {"role": "assistant", "content": "A limit on requests or tokens within a given time period. Exceeding it returns errors such as HTTP 429. Free tiers have particularly strict limits, and agents reach them easily because they call the LLM multiple times for one question."}]}',
    '{"messages": [{"role": "user", "content": "What is GitHub Codespaces?"}, {"role": "assistant", "content": "A cloud development environment provided by GitHub. Open VS Code in a browser and immediately use an environment configured by the repository. Personal accounts have a monthly free allowance; if you do not stop the Codespace when idle, it continues consuming that allowance."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain GitHub Codespaces."}, {"role": "assistant", "content": "A cloud development environment provided by GitHub. Open VS Code in a browser and immediately use an environment configured by the repository. Personal accounts have a monthly free allowance; if you do not stop the Codespace when idle, it continues consuming that allowance."}]}',
    '{"messages": [{"role": "user", "content": "What is devcontainer?"}, {"role": "assistant", "content": "A mechanism for defining a development environment as a container. Specify the Python version, extensions, and startup commands in .devcontainer/devcontainer.json so everyone opening Codespaces gets the same environment."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain devcontainer."}, {"role": "assistant", "content": "A mechanism for defining a development environment as a container. Specify the Python version, extensions, and startup commands in .devcontainer/devcontainer.json so everyone opening Codespaces gets the same environment."}]}',
    '{"messages": [{"role": "user", "content": "What is Codespaces secret?"}, {"role": "assistant", "content": "A secret value registered in your GitHub account settings. It is passed as an environment variable when you start Codespaces for a selected repository. When registering it, select the repository you will use (your own fork). If registered while a Codespace is running, restart the Codespace."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Codespaces secret."}, {"role": "assistant", "content": "A secret value registered in your GitHub account settings. It is passed as an environment variable when you start Codespaces for a selected repository. When registering it, select the repository you will use (your own fork). If registered while a Codespace is running, restart the Codespace."}]}',
    '{"messages": [{"role": "user", "content": "What is fork?"}, {"role": "assistant", "content": "Copying someone else\'s entire repository to your own account. In this course, fork the course repository and work and submit within your own fork."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain fork."}, {"role": "assistant", "content": "Copying someone else\'s entire repository to your own account. In this course, fork the course repository and work and submit within your own fork."}]}',
    '{"messages": [{"role": "user", "content": "What is Branch?"}, {"role": "assistant", "content": "A mechanism for separating streams of work. Try changes without breaking main, then merge them into main through a PR once ready. In this course, create a working branch such as lec01 for each lecture."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Branch."}, {"role": "assistant", "content": "A mechanism for separating streams of work. Try changes without breaking main, then merge them into main through a PR once ready. In this course, create a working branch such as lec01 for each lecture."}]}',
    '{"messages": [{"role": "user", "content": "What is Pull Request (PR)?"}, {"role": "assistant", "content": "A mechanism for requesting that changes from one branch be merged into another. It supports reviewing differences, comments, and displaying automated test results. Submit work for this course as a PR from your working branch to main within your own fork."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Pull Request (PR)."}, {"role": "assistant", "content": "A mechanism for requesting that changes from one branch be merged into another. It supports reviewing differences, comments, and displaying automated test results. Submit work for this course as a PR from your working branch to main within your own fork."}]}',
    '{"messages": [{"role": "user", "content": "What is GitHub Copilot?"}, {"role": "assistant", "content": "GitHub\'s AI coding assistance. It supports code completion in the editor, questions in chat, and editing multiple files in Agent mode. Copilot Free limits completion counts and chat usage."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain GitHub Copilot."}, {"role": "assistant", "content": "GitHub\'s AI coding assistance. It supports code completion in the editor, questions in chat, and editing multiple files in Agent mode. Copilot Free limits completion counts and chat usage."}]}',
    '{"messages": [{"role": "user", "content": "What is Code completion?"}, {"role": "assistant", "content": "A feature in which AI suggests code to follow partially written code or comments. Accept suggestions with Tab or a similar action. Suggestions may be incorrect, so read and understand them before accepting."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Code completion."}, {"role": "assistant", "content": "A feature in which AI suggests code to follow partially written code or comments. Accept suggestions with Tab or a similar action. Suggestions may be incorrect, so read and understand them before accepting."}]}',
    '{"messages": [{"role": "user", "content": "What is Coding agent?"}, {"role": "assistant", "content": "AI that autonomously carries out requested programming work over multiple steps, using tools to read and write files and run commands. Examples include Claude Code, Copilot Agent mode, and Pi. Internally, it has the same LLM + loop + tools structure as an AI agent."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Coding agent."}, {"role": "assistant", "content": "AI that autonomously carries out requested programming work over multiple steps, using tools to read and write files and run commands. Examples include Claude Code, Copilot Agent mode, and Pi. Internally, it has the same LLM + loop + tools structure as an AI agent."}]}',
    '{"messages": [{"role": "user", "content": "What is Pretraining?"}, {"role": "assistant", "content": "The initial stage of training a model to predict the next token using large amounts of general-purpose text. Much of its language knowledge and general common sense is acquired here."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Pretraining."}, {"role": "assistant", "content": "The initial stage of training a model to predict the next token using large amounts of general-purpose text. Much of its language knowledge and general common sense is acquired here."}]}',
    '{"messages": [{"role": "user", "content": "What is Fine-tuning?"}, {"role": "assistant", "content": "Further training a pretrained model on data suited to a specific task or format. It is suitable for teaching output format, tone, or behavior in a specific domain. For up-to-date facts, consulting external data through tools or RAG is often more suitable."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Fine-tuning."}, {"role": "assistant", "content": "Further training a pretrained model on data suited to a specific task or format. It is suitable for teaching output format, tone, or behavior in a specific domain. For up-to-date facts, consulting external data through tools or RAG is often more suitable."}]}',
    '{"messages": [{"role": "user", "content": "What is Supervised Fine-Tuning (SFT)?"}, {"role": "assistant", "content": "A method of further training a model using pairs of inputs and desired outputs, such as questions and model answers. A representative method for teaching instruction-following behavior."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Supervised Fine-Tuning (SFT)."}, {"role": "assistant", "content": "A method of further training a model using pairs of inputs and desired outputs, such as questions and model answers. A representative method for teaching instruction-following behavior."}]}',
    '{"messages": [{"role": "user", "content": "What is PEFT?"}, {"role": "assistant", "content": "An umbrella term for methods that train only a small subset of model parameters or a small set of added parameters instead of all parameters. This can greatly reduce GPU memory and storage requirements. LoRA is a representative example."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain PEFT."}, {"role": "assistant", "content": "An umbrella term for methods that train only a small subset of model parameters or a small set of added parameters instead of all parameters. This can greatly reduce GPU memory and storage requirements. LoRA is a representative example."}]}',
    '{"messages": [{"role": "user", "content": "What is LoRA?"}, {"role": "assistant", "content": "A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain LoRA."}, {"role": "assistant", "content": "A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA."}]}',
    '{"messages": [{"role": "user", "content": "What is Rank (LoRA r)?"}, {"role": "assistant", "content": "A hyperparameter that determines the size of the low-rank matrices used in LoRA. A larger value increases expressive capacity, but also increases the number of trainable parameters and memory use."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Rank (LoRA r)."}, {"role": "assistant", "content": "A hyperparameter that determines the size of the low-rank matrices used in LoRA. A larger value increases expressive capacity, but also increases the number of trainable parameters and memory use."}]}',
    '{"messages": [{"role": "user", "content": "What is QLoRA?"}, {"role": "assistant", "content": "A method that saves memory by quantizing base-model weights to, for example, 4 bits, then trains with LoRA. This lets smaller GPUs handle larger models. In the Lecture 02 exercises, the model fits in GPU memory, so we train with LoRA and compare memory and output differences when loading in 4-bit in the advanced section."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain QLoRA."}, {"role": "assistant", "content": "A method that saves memory by quantizing base-model weights to, for example, 4 bits, then trains with LoRA. This lets smaller GPUs handle larger models. In the Lecture 02 exercises, the model fits in GPU memory, so we train with LoRA and compare memory and output differences when loading in 4-bit in the advanced section."}]}',
    '{"messages": [{"role": "user", "content": "What is Quantization?"}, {"role": "assistant", "content": "A technique for representing model weights with fewer bits (for example, from 16 bits to 4 bits) to reduce memory use and computation. Accuracy may decrease slightly."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Quantization."}, {"role": "assistant", "content": "A technique for representing model weights with fewer bits (for example, from 16 bits to 4 bits) to reduce memory use and computation. Accuracy may decrease slightly."}]}',
    '{"messages": [{"role": "user", "content": "What is Learning rate?"}, {"role": "assistant", "content": "A hyperparameter that determines how far parameters move in one update. Too large a value makes training unstable; too small a value slows progress."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Learning rate."}, {"role": "assistant", "content": "A hyperparameter that determines how far parameters move in one update. Too large a value makes training unstable; too small a value slows progress."}]}',
    '{"messages": [{"role": "user", "content": "What is Epoch?"}, {"role": "assistant", "content": "One pass through the entire training dataset is one epoch. Too many epochs can cause overfitting."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Epoch."}, {"role": "assistant", "content": "One pass through the entire training dataset is one epoch. Too many epochs can cause overfitting."}]}',
    '{"messages": [{"role": "user", "content": "What is Overfitting?"}, {"role": "assistant", "content": "A state in which the model fits training data excessively and performs worse on unseen inputs. Detect it by measuring performance on validation data not used for training."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Overfitting."}, {"role": "assistant", "content": "A state in which the model fits training data excessively and performs worse on unseen inputs. Detect it by measuring performance on validation data not used for training."}]}',
    '{"messages": [{"role": "user", "content": "What is Hugging Face Hub?"}, {"role": "assistant", "content": "A platform for sharing and downloading public models and datasets. Used to obtain open-weight models and publish LoRA adapters."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Hugging Face Hub."}, {"role": "assistant", "content": "A platform for sharing and downloading public models and datasets. Used to obtain open-weight models and publish LoRA adapters."}]}',
    '{"messages": [{"role": "user", "content": "What is AI agent?"}, {"role": "assistant", "content": "A system that achieves a goal through a loop in which an LLM decides to call tools, examines the results, and decides its next action. Its basic structure is \\"model + loop + tools\\"."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain AI agent."}, {"role": "assistant", "content": "A system that achieves a goal through a loop in which an LLM decides to call tools, examines the results, and decides its next action. Its basic structure is \\"model + loop + tools\\"."}]}',
    '{"messages": [{"role": "user", "content": "What is Agent loop?"}, {"role": "assistant", "content": "The process of passing input to the model → executing any tool calls returned by the model → returning results to the model, repeated until the model gives a final answer. Multiple LLM calls for one question have a substantial effect on time, charges, and rate limits."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Agent loop."}, {"role": "assistant", "content": "The process of passing input to the model → executing any tool calls returned by the model → returning results to the model, repeated until the model gives a final answer. Multiple LLM calls for one question have a substantial effect on time, charges, and rate limits."}]}',
    '{"messages": [{"role": "user", "content": "What is Tool (Tool use / Function calling)?"}, {"role": "assistant", "content": "A function made available for the model to call. The model looks at its name, description, and argument types, chooses arguments when needed, and requests a call; the program executes it. Used for search, calculations, file operations, external API calls, and more."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Tool (Tool use / Function calling)."}, {"role": "assistant", "content": "A function made available for the model to call. The model looks at its name, description, and argument types, chooses arguments when needed, and requests a call; the program executes it. Used for search, calculations, file operations, external API calls, and more."}]}',
    '{"messages": [{"role": "user", "content": "What is Tool description?"}, {"role": "assistant", "content": "Text that explains a tool\'s purpose and argument meanings to the model. In Pydantic AI, it is generated automatically from the function docstring and type hints. Since the model reads it to decide whether and how to call a tool, its wording can greatly affect behavior."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Tool description."}, {"role": "assistant", "content": "Text that explains a tool\'s purpose and argument meanings to the model. In Pydantic AI, it is generated automatically from the function docstring and type hints. Since the model reads it to decide whether and how to call a tool, its wording can greatly affect behavior."}]}',
    '{"messages": [{"role": "user", "content": "What is Structured output?"}, {"role": "assistant", "content": "A mechanism for receiving model outputs according to specified types or a JSON schema. This makes programmatic postprocessing easier. In Pydantic AI, specify a Pydantic model in output_type."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Structured output."}, {"role": "assistant", "content": "A mechanism for receiving model outputs according to specified types or a JSON schema. This makes programmatic postprocessing easier. In Pydantic AI, specify a Pydantic model in output_type."}]}',
    '{"messages": [{"role": "user", "content": "What is Harness?"}, {"role": "assistant", "content": "A program that surrounds an LLM with a loop, tools, a user interface (UI), and so on to enable actual work. Claude Code and Pi are examples. Pi is known as a thin harness with just four built-in tools: read, write, edit, and bash."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Harness."}, {"role": "assistant", "content": "A program that surrounds an LLM with a loop, tools, a user interface (UI), and so on to enable actual work. Claude Code and Pi are examples. Pi is known as a thin harness with just four built-in tools: read, write, edit, and bash."}]}',
    '{"messages": [{"role": "user", "content": "What is Pi?"}, {"role": "assistant", "content": "An open-source (MIT-licensed) coding agent that runs in the terminal. It has minimal features and a minimal system prompt, designed for users to extend. Version 1.0 was released in October 2026. Its npm package name is @earendil-works/pi-coding-agent."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Pi."}, {"role": "assistant", "content": "An open-source (MIT-licensed) coding agent that runs in the terminal. It has minimal features and a minimal system prompt, designed for users to extend. Version 1.0 was released in October 2026. Its npm package name is @earendil-works/pi-coding-agent."}]}',
    '{"messages": [{"role": "user", "content": "What is Pydantic AI?"}, {"role": "assistant", "content": "A framework for building agents in Python. It uses Pydantic types to define tool arguments and outputs and handles many models, such as Gemini and OpenAI, with the same code style. Test models let you check behavior without API calls. It is the main framework for Lecture 03."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Pydantic AI."}, {"role": "assistant", "content": "A framework for building agents in Python. It uses Pydantic types to define tool arguments and outputs and handles many models, such as Gemini and OpenAI, with the same code style. Test models let you check behavior without API calls. It is the main framework for Lecture 03."}]}',
    '{"messages": [{"role": "user", "content": "What is ADK (Agent Development Kit)?"}, {"role": "assistant", "content": "An open-source agent development framework released by Google. It integrates closely with Gemini, and adk web lets you visually follow tool calls. Version 2.0 introduces breaking changes from the 1.x series. This course covers it in the Advanced homework."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain ADK (Agent Development Kit)."}, {"role": "assistant", "content": "An open-source agent development framework released by Google. It integrates closely with Gemini, and adk web lets you visually follow tool calls. Version 2.0 introduces breaking changes from the 1.x series. This course covers it in the Advanced homework."}]}',
    '{"messages": [{"role": "user", "content": "What is MCP (Model Context Protocol)?"}, {"role": "assistant", "content": "An open standard protocol connecting AI applications to external tools and data sources. Tools exposed through an MCP server can be used in a common way from compatible agents and editors (MCP clients). In Lecture 05, you will experience this by turning glossary search into an MCP server."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain MCP (Model Context Protocol)."}, {"role": "assistant", "content": "An open standard protocol connecting AI applications to external tools and data sources. Tools exposed through an MCP server can be used in a common way from compatible agents and editors (MCP clients). In Lecture 05, you will experience this by turning glossary search into an MCP server."}]}',
    '{"messages": [{"role": "user", "content": "What is Guardrail?"}, {"role": "assistant", "content": "A constraint or check designed to prevent undesirable agent behavior or outputs. Methods include rules in instructions, restricting tool permissions, checking outputs, and requesting human confirmation before important operations."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Guardrail."}, {"role": "assistant", "content": "A constraint or check designed to prevent undesirable agent behavior or outputs. Methods include rules in instructions, restricting tool permissions, checking outputs, and requesting human confirmation before important operations."}]}',
    '{"messages": [{"role": "user", "content": "What is Human-in-the-loop?"}, {"role": "assistant", "content": "Incorporating human confirmation or approval into agent processing. Used before operations that are difficult to undo, such as writing or sending data. In Pydantic AI, tools with requires_approval=True support this."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Human-in-the-loop."}, {"role": "assistant", "content": "Incorporating human confirmation or approval into agent processing. Used before operations that are difficult to undo, such as writing or sending data. In Pydantic AI, tools with requires_approval=True support this."}]}',
    '{"messages": [{"role": "user", "content": "What is TestModel?"}, {"role": "assistant", "content": "A fake model included in Pydantic AI for testing. It calls tools and returns dummy answers without calling a real LLM. It lets you test agent wiring (whether tools are registered, called, and return results) without an API key or usage allowance."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain TestModel."}, {"role": "assistant", "content": "A fake model included in Pydantic AI for testing. It calls tools and returns dummy answers without calling a real LLM. It lets you test agent wiring (whether tools are registered, called, and return results) without an API key or usage allowance."}]}',
    '{"messages": [{"role": "user", "content": "What is 3 homework levels?"}, {"role": "assistant", "content": "The structure of AIE2026 homework. There are three levels: \\"Basic (review lecture content)\\", \\"Implementation (write your own code)\\", and \\"Advanced (research and experimentation)\\". Completing any one level completes the homework for that lecture. As a rule, it is not graded."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain 3 homework levels."}, {"role": "assistant", "content": "The structure of AIE2026 homework. There are three levels: \\"Basic (review lecture content)\\", \\"Implementation (write your own code)\\", and \\"Advanced (research and experimentation)\\". Completing any one level completes the homework for that lecture. As a rule, it is not graded."}]}',
    '{"messages": [{"role": "user", "content": "What is answer.md?"}, {"role": "assistant", "content": "The file for recording exercise and homework results for each AIE2026 lecture. Located at lectures/NN/answer.md. Record the level attempted, what you did, results, and observations, then submit by opening a PR from your working branch to main in your own fork."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain answer.md."}, {"role": "assistant", "content": "The file for recording exercise and homework results for each AIE2026 lecture. Located at lectures/NN/answer.md. Record the level attempted, what you did, results, and observations, then submit by opening a PR from your working branch to main in your own fork."}]}',
    '{"messages": [{"role": "user", "content": "What is lecNN-start tag?"}, {"role": "assistant", "content": "A Git tag on the starting code for each lecture\'s exercises in the AIE2026 course repository (for example, lec03-start). Even if you have not finished the previous lecture, creating a branch from this tag lets you start the current lecture\'s exercises."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain lecNN-start tag."}, {"role": "assistant", "content": "A Git tag on the starting code for each lecture\'s exercises in the AIE2026 course repository (for example, lec03-start). Even if you have not finished the previous lecture, creating a branch from this tag lets you start the current lecture\'s exercises."}]}',
    '{"messages": [{"role": "user", "content": "What is Default model?"}, {"role": "assistant", "content": "The standard model for AIE2026 exercises and homework is gemini-3.5-flash-lite, chosen with free-tier rate limits and stable course operation in mind. You may switch models with the AIE_MODEL environment variable or use your own API subscription."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Default model."}, {"role": "assistant", "content": "The standard model for AIE2026 exercises and homework is gemini-3.5-flash-lite, chosen with free-tier rate limits and stable course operation in mind. You may switch models with the AIE_MODEL environment variable or use your own API subscription."}]}',
    '{"messages": [{"role": "user", "content": "What is External tool?"}, {"role": "assistant", "content": "Using capabilities outside the agent program, such as web retrieval, file operations, databases, and third-party APIs, as tools. Lecture 03 function tools run within the same process, whereas external tools are called across processes or machines. MCP is a common protocol for this."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain External tool."}, {"role": "assistant", "content": "Using capabilities outside the agent program, such as web retrieval, file operations, databases, and third-party APIs, as tools. Lecture 03 function tools run within the same process, whereas external tools are called across processes or machines. MCP is a common protocol for this."}]}',
    '{"messages": [{"role": "user", "content": "What is MCP server?"}, {"role": "assistant", "content": "The program that exposes tools (and resources or prompts) through MCP. It exposes tool names, descriptions, and argument schemas, executes tools in response to client requests, and returns results. In this course, glossary search becomes an MCP server (app/mcp_server.py). Many ready-made servers are available, including fetch, filesystem, and GitHub."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain MCP server."}, {"role": "assistant", "content": "The program that exposes tools (and resources or prompts) through MCP. It exposes tool names, descriptions, and argument schemas, executes tools in response to client requests, and returns results. In this course, glossary search becomes an MCP server (app/mcp_server.py). Many ready-made servers are available, including fetch, filesystem, and GitHub."}]}',
    '{"messages": [{"role": "user", "content": "What is MCP client?"}, {"role": "assistant", "content": "The component that connects to MCP servers, passes their tools to the model, and calls tools as directed by the model. Agent programs (Pydantic AI\'s MCPToolset) and editors or applications such as Copilot Chat, Claude Desktop, and Cursor can serve as clients. A benefit of MCP is using the same server from different clients."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain MCP client."}, {"role": "assistant", "content": "The component that connects to MCP servers, passes their tools to the model, and calls tools as directed by the model. Agent programs (Pydantic AI\'s MCPToolset) and editors or applications such as Copilot Chat, Claude Desktop, and Cursor can serve as clients. A benefit of MCP is using the same server from different clients."}]}',
    '{"messages": [{"role": "user", "content": "What is Transport (stdio / Streamable HTTP)?"}, {"role": "assistant", "content": "The communication channel between an MCP client and server. With stdio, the client starts the server as a subprocess and exchanges messages through standard input/output (simple and used on the same machine). With Streamable HTTP, the server listens at a URL, allowing connections from other processes or machines (access scope and authentication must be considered)."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Transport (stdio / Streamable HTTP)."}, {"role": "assistant", "content": "The communication channel between an MCP client and server. With stdio, the client starts the server as a subprocess and exchanges messages through standard input/output (simple and used on the same machine). With Streamable HTTP, the server listens at a URL, allowing connections from other processes or machines (access scope and authentication must be considered)."}]}',
    '{"messages": [{"role": "user", "content": "What is mcpServers configuration?"}, {"role": "assistant", "content": "A configuration file listing MCP servers to use. For each server, specify a name and a startup command (stdio) or destination URL (HTTP). Claude Desktop, Cursor, and Pydantic AI read the same mcpServers format (VS Code / Copilot Chat uses servers). This course uses app/mcp_servers.json. Adding tools with just a few configuration lines is central to the MCP experience."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain mcpServers configuration."}, {"role": "assistant", "content": "A configuration file listing MCP servers to use. For each server, specify a name and a startup command (stdio) or destination URL (HTTP). Claude Desktop, Cursor, and Pydantic AI read the same mcpServers format (VS Code / Copilot Chat uses servers). This course uses app/mcp_servers.json. Adding tools with just a few configuration lines is central to the MCP experience."}]}',
    '{"messages": [{"role": "user", "content": "What is Tool namespace?"}, {"role": "assistant", "content": "Adding a prefix such as \\"server_name_tool_name\\" to distinguish identically named tools when using multiple MCP servers. This course\'s app/agent.py automatically adds the server name from the configuration (for example, glossary_search_glossary; Pydantic AI\'s load_mcp_toolsets does the same). More tools mean more descriptions sent to the model, affecting tool-selection accuracy and token counts."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Tool namespace."}, {"role": "assistant", "content": "Adding a prefix such as \\"server_name_tool_name\\" to distinguish identically named tools when using multiple MCP servers. This course\'s app/agent.py automatically adds the server name from the configuration (for example, glossary_search_glossary; Pydantic AI\'s load_mcp_toolsets does the same). More tools mean more descriptions sent to the model, affecting tool-selection accuracy and token counts."}]}',
    '{"messages": [{"role": "user", "content": "What is fetch server?"}, {"role": "assistant", "content": "A ready-made MCP server that retrieves a specified URL and returns the page body as Markdown (written in Python; started with uvx mcp-server-fetch). This course uses it as an example of \\"adding an external tool just through configuration\\". It actually accesses URLs chosen by the model, so users must understand where it sends data and what it sends."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain fetch server."}, {"role": "assistant", "content": "A ready-made MCP server that retrieves a specified URL and returns the page body as Markdown (written in Python; started with uvx mcp-server-fetch). This course uses it as an example of \\"adding an external tool just through configuration\\". It actually accesses URLs chosen by the model, so users must understand where it sends data and what it sends."}]}',
]

EVAL_SET = {'abstain_answer': 'That term is not in the course glossary.',
 'train_absent_terms': ['Direct Preference Optimization',
                        'speculative decoding',
                        'KV cache',
                        'perplexity',
                        'diffusion model',
                        'batch normalization',
                        'gradient checkpointing',
                        'BLEU score'],
 'items': [{'group': 'in_glossary',
            'question': 'What is LLM (Large Language Model)?',
            'term_id': 'llm',
            'term': 'LLM (Large Language Model)',
            'gold': 'A model trained on large amounts of text that generates text by predicting the tokens following its input. It can '
                    'handle many tasks, including question answering, summarization, and code generation, simply by changing the '
                    'instructions (prompt).',
            'must_include': ['predicting', 'summarization'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'Briefly explain Context window.',
            'term_id': 'context-window',
            'term': 'Context window',
            'gold': 'The maximum total number of input and output tokens an LLM can handle in one call. Conversation history and tool '
                    'results must also fit within this limit, making the choice of what to pass to the model important.',
            'must_include': ['one call', 'tool results'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'What is Hallucination?',
            'term_id': 'hallucination',
            'term': 'Hallucination',
            'gold': 'The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely '
                    'when asked about things the model does not know, such as recent information or organization- or course-specific '
                    'details. Mitigations include consulting external data through tools, asking for evidence, and detecting errors '
                    'through evaluation.',
            'must_include': ['course-specific', 'external data'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'What is LoRA?',
            'term_id': 'lora',
            'term': 'LoRA',
            'gold': 'A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices '
                    '(low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) '
                    'in small files. The Lecture 02 exercises cover LoRA.',
            'must_include': ['low-rank', 'adapters', 'Lecture 02'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'Briefly explain QLoRA.',
            'term_id': 'qlora',
            'term': 'QLoRA',
            'gold': 'A method that saves memory by quantizing base-model weights to, for example, 4 bits, then trains with LoRA. This lets '
                    'smaller GPUs handle larger models. In the Lecture 02 exercises, the model fits in GPU memory, so we train with LoRA '
                    'and compare memory and output differences when loading in 4-bit in the advanced section.',
            'must_include': ['4 bits', 'smaller GPUs'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'What is Overfitting?',
            'term_id': 'overfitting',
            'term': 'Overfitting',
            'gold': 'A state in which the model fits training data excessively and performs worse on unseen inputs. Detect it by measuring '
                    'performance on validation data not used for training.',
            'must_include': ['validation data', 'unseen'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'What is MCP (Model Context Protocol)?',
            'term_id': 'mcp',
            'term': 'MCP (Model Context Protocol)',
            'gold': 'An open standard protocol connecting AI applications to external tools and data sources. Tools exposed through an MCP '
                    'server can be used in a common way from compatible agents and editors (MCP clients). In Lecture 05, you will '
                    'experience this by turning glossary search into an MCP server.',
            'must_include': ['open standard', 'Lecture 05'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'Briefly explain Guardrail.',
            'term_id': 'guardrail',
            'term': 'Guardrail',
            'gold': 'A constraint or check designed to prevent undesirable agent behavior or outputs. Methods include rules in '
                    'instructions, restricting tool permissions, checking outputs, and requesting human confirmation before important '
                    'operations.',
            'must_include': ['tool permissions', 'human confirmation'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'What is Harness?',
            'term_id': 'harness',
            'term': 'Harness',
            'gold': 'A program that surrounds an LLM with a loop, tools, a user interface (UI), and so on to enable actual work. Claude '
                    'Code and Pi are examples. Pi is known as a thin harness with just four built-in tools: read, write, edit, and bash.',
            'must_include': ['Claude Code', 'four built-in tools'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'What is Pi?',
            'term_id': 'pi',
            'term': 'Pi',
            'gold': 'An open-source (MIT-licensed) coding agent that runs in the terminal. It has minimal features and a minimal system '
                    'prompt, designed for users to extend. Version 1.0 was released in October 2026. Its npm package name is '
                    '@earendil-works/pi-coding-agent.',
            'must_include': ['MIT', 'terminal', '@earendil-works/pi-coding-agent'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'What is Default model?',
            'term_id': 'default-model',
            'term': 'Default model',
            'gold': 'The standard model for AIE2026 exercises and homework is gemini-3.5-flash-lite, chosen with free-tier rate limits and '
                    'stable course operation in mind. You may switch models with the AIE_MODEL environment variable or use your own API '
                    'subscription.',
            'must_include': ['gemini-3.5-flash-lite', 'AIE_MODEL'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'Briefly explain answer.md.',
            'term_id': 'answer-md',
            'term': 'answer.md',
            'gold': 'The file for recording exercise and homework results for each AIE2026 lecture. Located at lectures/NN/answer.md. '
                    'Record the level attempted, what you did, results, and observations, then submit by opening a PR from your working '
                    'branch to main in your own fork.',
            'must_include': ['lectures/NN/answer.md', 'working branch'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'paraphrased',
            'question': 'What does HITL mean in agent design?',
            'term_id': 'human-in-the-loop',
            'term': 'Human-in-the-loop',
            'gold': 'Incorporating human confirmation or approval into agent processing. Used before operations that are difficult to '
                    'undo, such as writing or sending data. In Pydantic AI, tools with requires_approval=True support this.',
            'must_include': ['difficult to undo', 'requires_approval'],
            'why': "Alias (acronym) plus extra context words; also contains the generic alias 'agent', which tempts a naive search."},
           {'group': 'paraphrased',
            'question': 'In LoRA, what does the r hyperparameter control?',
            'term_id': 'lora-rank',
            'term': 'Rank (LoRA r)',
            'gold': 'A hyperparameter that determines the size of the low-rank matrices used in LoRA. A larger value increases expressive '
                    'capacity, but also increases the number of trainable parameters and memory use.',
            'must_include': ['expressive capacity', 'trainable parameters'],
            'why': "Asks about the one-letter alias 'r' inside a LoRA question; tests whether the right entry (rank, not LoRA) is chosen."},
           {'group': 'paraphrased',
            'question': 'Why would I get a 429 error from an LLM API?',
            'term_id': 'rate-limit',
            'term': 'Rate limit',
            'gold': 'A limit on requests or tokens within a given time period. Exceeding it returns errors such as HTTP 429. Free tiers '
                    'have particularly strict limits, and agents reach them easily because they call the LLM multiple times for one '
                    'question.',
            'must_include': ['free tiers', 'multiple times'],
            'why': "Symptom-style question: the alias '429 error' appears, but the term 'rate limit' never does."},
           {'group': 'paraphrased',
            'question': 'What does Low-Rank Adaptation do to a pretrained model?',
            'term_id': 'lora',
            'term': 'LoRA',
            'gold': 'A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices '
                    '(low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) '
                    'in small files. The Lecture 02 exercises cover LoRA.',
            'must_include': ['adapters', 'Lecture 02'],
            'why': 'Long-form alias with a new question frame; course-specific keywords check it is the glossary answer, not general '
                   'knowledge.'},
           {'group': 'paraphrased',
            'question': 'How do I pass a Codespaces secret to my code, and what if my Codespace is already running?',
            'term_id': 'codespaces-secret',
            'term': 'Codespaces secret',
            'gold': 'A secret value registered in your GitHub account settings. It is passed as an environment variable when you start '
                    'Codespaces for a selected repository. When registering it, select the repository you will use (your own fork). If '
                    'registered while a Codespace is running, restart the Codespace.',
            'must_include': ['environment variable', 'restart', 'own fork'],
            'why': "Two-part practical question that overlaps with the more generic 'Codespaces' entry."},
           {'group': 'paraphrased',
            'question': 'What goes into devcontainer.json?',
            'term_id': 'devcontainer',
            'term': 'devcontainer',
            'gold': 'A mechanism for defining a development environment as a container. Specify the Python version, extensions, and '
                    'startup commands in .devcontainer/devcontainer.json so everyone opening Codespaces gets the same environment.',
            'must_include': ['Python version', 'startup commands'],
            'why': "Asks via the file-name alias and about contents rather than 'what is'."},
           {'group': 'paraphrased',
            'question': 'What is instruction tuning?',
            'term_id': 'sft',
            'term': 'Supervised Fine-Tuning (SFT)',
            'gold': 'A method of further training a model using pairs of inputs and desired outputs, such as questions and model answers. '
                    'A representative method for teaching instruction-following behavior.',
            'must_include': ['pairs', 'instruction-following'],
            'why': 'Synonym alias that never names SFT; general knowledge may give a different (broader) definition.'},
           {'group': 'paraphrased',
            'question': 'Walk me through the tool-calling loop of an agent.',
            'term_id': 'agent-loop',
            'term': 'Agent loop',
            'gold': 'The process of passing input to the model → executing any tool calls returned by the model → returning results to the '
                    'model, repeated until the model gives a final answer. Multiple LLM calls for one question have a substantial effect '
                    'on time, charges, and rate limits.',
            'must_include': ['final answer', 'rate limits'],
            'why': "Imperative wording with the alias 'tool-calling loop'; competes with 'tool' and 'agent' entries."},
           {'group': 'paraphrased',
            'question': 'Which model do the AIE2026 exercises use by default?',
            'term_id': 'default-model',
            'term': 'Default model',
            'gold': 'The standard model for AIE2026 exercises and homework is gemini-3.5-flash-lite, chosen with free-tier rate limits and '
                    'stable course operation in mind. You may switch models with the AIE_MODEL environment variable or use your own API '
                    'subscription.',
            'must_include': ['gemini-3.5-flash-lite', 'AIE_MODEL'],
            'why': 'Purely course-specific fact asked without the term or any alias; only memorized knowledge can answer it.'},
           {'group': 'paraphrased',
            'question': 'What should I avoid doing with my API key, and how is it read in this course?',
            'term_id': 'api-key',
            'term': 'API key',
            'gold': 'A secret string that identifies the person calling an API. If it leaks, someone else can consume your usage '
                    'allowance, so do not put it directly in code, commit it, or paste it into chat. In this course, register it as a '
                    'Codespaces secret and read it through the GEMINI_API_KEY environment variable.',
            'must_include': ['commit', 'GEMINI_API_KEY'],
            'why': 'Advice-style question; GEMINI_API_KEY is a course-specific detail a generic answer would miss.'},
           {'group': 'paraphrased',
            'question': 'Explain quantisation of model weights.',
            'term_id': 'quantization',
            'term': 'Quantization',
            'gold': 'A technique for representing model weights with fewer bits (for example, from 16 bits to 4 bits) to reduce memory use '
                    'and computation. Accuracy may decrease slightly.',
            'must_include': ['fewer bits', '4 bits'],
            'why': "British spelling ('quantisation'): no exact match, tests the fuzzy fallback and spelling robustness."},
           {'group': 'paraphrased',
            'question': 'Why do LLMs sometimes confidently make up facts?',
            'term_id': 'hallucination',
            'term': 'Hallucination',
            'gold': 'The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely '
                    'when asked about things the model does not know, such as recent information or organization- or course-specific '
                    'details. Mitigations include consulting external data through tools, asking for evidence, and detecting errors '
                    'through evaluation.',
            'must_include': ['course-specific', 'external data'],
            'why': 'Describes the phenomenon without naming it; keyword search has nothing to match.'},
           {'group': 'paraphrased',
            'question': 'How do I avoid name collisions between tools from multiple MCP servers?',
            'term_id': 'tool-namespace',
            'term': 'Tool namespace',
            'gold': 'Adding a prefix such as "server_name_tool_name" to distinguish identically named tools when using multiple MCP '
                    "servers. This course's app/agent.py automatically adds the server name from the configuration (for example, "
                    "glossary_search_glossary; Pydantic AI's load_mcp_toolsets does the same). More tools mean more descriptions sent to "
                    'the model, affecting tool-selection accuracy and token counts.',
            'must_include': ['prefix', 'server_name_tool_name'],
            'why': "Plural of the alias 'name collision' plus the competing term 'MCP'; tests problem-to-term mapping."},
           {'group': 'absent',
            'question': 'What is RAG (retrieval-augmented generation)?',
            'term': 'RAG',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Very common AI term; closely related to the course's glossary-search agent but not a glossary entry."},
           {'group': 'absent',
            'question': 'Can you explain RLHF?',
            'term': 'RLHF',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': 'Training-related acronym that sits next to SFT/fine-tuning, so the model may borrow those definitions.'},
           {'group': 'absent',
            'question': 'What is a vector database used for?',
            'term': 'vector database',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': 'Infrastructure term with no lexical overlap with glossary keys.'},
           {'group': 'absent',
            'question': 'What is an embedding in machine learning?',
            'term': 'embedding',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': 'Generic ML term the base model knows well; tests abstaining despite confident knowledge.'},
           {'group': 'absent',
            'question': 'How does beam search decoding work?',
            'term': 'beam search',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Contains the word 'search' (the course builds a glossary search) but is a decoding method."},
           {'group': 'absent',
            'question': 'What is knowledge distillation?',
            'term': 'knowledge distillation',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': 'Training technique adjacent to fine-tuning/LoRA.'},
           {'group': 'absent',
            'question': 'What is a Mixture of Experts model?',
            'term': 'Mixture of Experts',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Architecture term; 'model' is generic and not a glossary key."},
           {'group': 'absent',
            'question': 'What is prompt injection?',
            'term': 'prompt injection',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Deliberately contains the glossary term 'prompt': a substring-based search will wrongly assert the Prompt definition."},
           {'group': 'absent',
            'question': 'What does top-p sampling do?',
            'term': 'top-p',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Generation setting adjacent to the glossary's 'temperature' entry."},
           {'group': 'absent',
            'question': 'What is chain-of-thought prompting?',
            'term': 'chain-of-thought',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "'prompting' is close to 'prompt'; tests the fuzzy-match threshold."}]}

# ---------------------------------------------------------------- 2. training set + checks (v2 cell, unchanged)
import hashlib, re

# 1) Training data = script output, verified against the recorded sha256
sft_text = "\n".join(SFT_JSONL_LINES) + "\n"
assert hashlib.sha256(sft_text.encode("utf-8")).hexdigest() == SFT_SHA256, "Embedded JSONL differs from the script output"
plain_records = [json.loads(line) for line in SFT_JSONL_LINES]

# Cross-check: regenerate with the same templates as scripts/make_sft_dataset.py from the embedded glossary
TERMS = GLOSSARY["terms"]
TEMPLATES = ["What is {term}?", "Briefly explain {term}."]   # Copied from scripts/make_sft_dataset.py

def qa(question: str, answer: str) -> dict:
    """One record in the same messages format as the script (and the original notebook's to_chat, without a system prompt)"""
    return {"messages": [{"role": "user", "content": question}, {"role": "assistant", "content": answer}]}

assert plain_records == [qa(tpl.format(term=t["term"]), t["definition"]) for t in TERMS for tpl in TEMPLATES]
print(f"sha256 OK | plain records={len(plain_records)} (= {len(TERMS)} terms x {len(TEMPLATES)} templates)")

# 2) Augmented variant: alias questions + abstention examples for training-only absent terms
ABSTAIN_ANSWER = EVAL_SET["abstain_answer"]          # "That term is not in the course glossary."
TRAIN_ABSENT_TERMS = EVAL_SET["train_absent_terms"]  # 8 terms, disjoint from the evaluation's absent terms
alias_records = [qa(f"What does {alias} mean?", t["definition"])
                 for t in TERMS for alias in t["aliases"]
                 if alias.lower() != t["term"].lower() and len(alias) > 1]   # Skip aliases equal to the term and the 1-letter alias "r"
abstain_records = [qa(tpl.format(term=x), ABSTAIN_ANSWER) for x in TRAIN_ABSENT_TERMS for tpl in TEMPLATES]
train_records = plain_records if VARIANT == "plain" else plain_records + alias_records + abstain_records

# 3) Checks
def norm(text: str) -> str:
    """Lowercase, turn punctuation into spaces, collapse whitespace (also used by the glossary search in section 6)"""
    return " ".join(re.sub(r"[^0-9a-z]+", " ", text.lower()).split())

def glossary_keys(t: dict) -> list:
    """Term, its parts around parentheses ('LLM (Large Language Model)' -> 'LLM', 'Large Language Model'), and aliases"""
    head, _, inner = t["term"].partition("(")
    parts = [t["term"], head] + inner.rstrip(")").split("/") + t["aliases"]
    return [p.strip() for p in parts if p.strip()]

GLOSSARY_KEYS = {norm(k) for t in TERMS for k in glossary_keys(t)}
EVAL_ITEMS = EVAL_SET["items"]
absent_terms = [it["term"] for it in EVAL_ITEMS if it["group"] == "absent"] + TRAIN_ABSENT_TERMS
for x in absent_terms:   # The absent term must not be (part of) any glossary term or alias
    clash = [k for k in GLOSSARY_KEYS if f" {norm(x)} " in f" {k} "]
    assert not clash, f"'{x}' appears in glossary keys {clash}"
shared_words = {x: [k for k in GLOSSARY_KEYS if f" {k} " in f" {norm(x)} "] for x in absent_terms}
print("Absent terms containing a glossary word (intentional search traps):", {x: k for x, k in shared_words.items() if k})
assert not set(map(norm, TRAIN_ABSENT_TERMS)) & set(map(norm, absent_terms[:-len(TRAIN_ABSENT_TERMS)])), "Train/eval absent terms overlap"

train_questions = {r["messages"][0]["content"] for r in plain_records + alias_records + abstain_records}
unseen = [it for it in EVAL_ITEMS if it["group"] != "in_glossary"]
overlap = [it["question"] for it in unseen if it["question"] in train_questions]
assert not overlap, f"Evaluation questions found in training data: {overlap}"
assert all(it["question"] in {r["messages"][0]["content"] for r in plain_records} for it in EVAL_ITEMS if it["group"] == "in_glossary")
assert len({r["messages"][0]["content"] for r in train_records}) == len(train_records), "Duplicate training questions"

groups = {g: sum(it["group"] == g for it in EVAL_ITEMS) for g in ("in_glossary", "paraphrased", "absent")}
print(f"variant={VARIANT} | train={len(train_records)} (plain {len(plain_records)} + alias {len(alias_records)} + abstain {len(abstain_records)}"
      f"{'' if VARIANT == 'augmented' else ', extras not used'}) | eval={len(EVAL_ITEMS)} {groups} | overlap=0 | absent terms checked={len(absent_terms)}")
print("One training example:", json.dumps(train_records[0], ensure_ascii=False)[:300])
_ = (WORK / f"train_{VARIANT}.jsonl").write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in train_records), encoding="utf-8")
_ = (WORK / "eval_set.json").write_text(json.dumps(EVAL_SET, ensure_ascii=False, indent=2), encoding="utf-8")

# ---------------------------------------------------------------- 2b. validation split and v3 evaluation set

EVAL_V3 = {'version': 'v3',
 'note': 'Separate from eval_set.json (35 questions) so earlier numbers stay comparable.',
 'abstain_answer': 'That term is not in the course glossary.',
 'primary_metric': {'alias': 'f1', 'made_up': 'abstain_rate'},
 'keyword_rule': 'for the alias group are hand-picked from the gold glossary definition: 1-2 specific, distinctive\n'
                 'content words per question (no generic words), none of which appear in the question itself. Token F1 against the\n'
                 'full definition is the primary metric for this group; keyword hit is secondary.',
 'items': [{'group': 'alias',
            'question': 'What is HITL?',
            'term_id': 'human-in-the-loop',
            'gold': 'Incorporating human confirmation or approval into agent processing. Used before operations that are difficult to '
                    'undo, such as writing or sending data. In Pydantic AI, tools with requires_approval=True support this.',
            'must_include': ['human', 'approval'],
            'why': 'acronym alias'},
           {'group': 'alias',
            'question': 'What is Parameter-Efficient Fine-Tuning?',
            'term_id': 'peft',
            'gold': 'An umbrella term for methods that train only a small subset of model parameters or a small set of added parameters '
                    'instead of all parameters. This can greatly reduce GPU memory and storage requirements. LoRA is a representative '
                    'example.',
            'must_include': ['subset', 'LoRA'],
            'why': 'full name of acronym term PEFT'},
           {'group': 'alias',
            'question': 'What is SFT?',
            'term_id': 'sft',
            'gold': 'A method of further training a model using pairs of inputs and desired outputs, such as questions and model answers. '
                    'A representative method for teaching instruction-following behavior.',
            'must_include': ['pairs', 'instruction-following'],
            'why': 'acronym part of term'},
           {'group': 'alias',
            'question': 'What is a PR?',
            'term_id': 'pull-request',
            'gold': 'A mechanism for requesting that changes from one branch be merged into another. It supports reviewing differences, '
                    'comments, and displaying automated test results. Submit work for this course as a PR from your working branch to main '
                    'within your own fork.',
            'must_include': ['branch', 'merged'],
            'why': 'acronym alias'},
           {'group': 'alias',
            'question': 'What is ADK?',
            'term_id': 'adk',
            'gold': 'An open-source agent development framework released by Google. It integrates closely with Gemini, and adk web lets '
                    'you visually follow tool calls. Version 2.0 introduces breaking changes from the 1.x series. This course covers it in '
                    'the Advanced homework.',
            'must_include': ['Google', 'Gemini'],
            'why': 'acronym part of term'},
           {'group': 'alias',
            'question': 'What is the Agent Development Kit?',
            'term_id': 'adk',
            'gold': 'An open-source agent development framework released by Google. It integrates closely with Gemini, and adk web lets '
                    'you visually follow tool calls. Version 2.0 introduces breaking changes from the 1.x series. This course covers it in '
                    'the Advanced homework.',
            'must_include': ['Google', 'Gemini'],
            'why': 'full-name alias'},
           {'group': 'alias',
            'question': 'What is the Model Context Protocol?',
            'term_id': 'mcp',
            'gold': 'An open standard protocol connecting AI applications to external tools and data sources. Tools exposed through an MCP '
                    'server can be used in a common way from compatible agents and editors (MCP clients). In Lecture 05, you will '
                    'experience this by turning glossary search into an MCP server.',
            'must_include': ['open standard', 'external tools'],
            'why': 'full-name alias'},
           {'group': 'alias',
            'question': 'What is Quantized LoRA?',
            'term_id': 'qlora',
            'gold': 'A method that saves memory by quantizing base-model weights to, for example, 4 bits, then trains with LoRA. This lets '
                    'smaller GPUs handle larger models. In the Lecture 02 exercises, the model fits in GPU memory, so we train with LoRA '
                    'and compare memory and output differences when loading in 4-bit in the advanced section.',
            'must_include': ['4 bits', 'memory'],
            'why': 'full name of QLoRA'},
           {'group': 'alias',
            'question': 'What is an Application Programming Interface?',
            'term_id': 'api',
            'gold': 'An interface through which a program calls the functionality of another service. In this course, we use an LLM by '
                    'calling the Gemini API from Python.',
            'must_include': ['functionality', 'service'],
            'why': 'full name of API'},
           {'group': 'alias',
            'question': 'What is a large language model?',
            'term_id': 'llm',
            'gold': 'A model trained on large amounts of text that generates text by predicting the tokens following its input. It can '
                    'handle many tasks, including question answering, summarization, and code generation, simply by changing the '
                    'instructions (prompt).',
            'must_include': ['tokens', 'predicting'],
            'why': 'full-name alias'},
           {'group': 'alias',
            'question': 'What is the HF Hub?',
            'term_id': 'huggingface-hub',
            'gold': 'A platform for sharing and downloading public models and datasets. Used to obtain open-weight models and publish LoRA '
                    'adapters.',
            'must_include': ['models', 'datasets'],
            'why': 'short alias'},
           {'group': 'alias',
            'question': 'What does lr stand for in the training settings?',
            'term_id': 'learning-rate',
            'gold': 'A hyperparameter that determines how far parameters move in one update. Too large a value makes training unstable; '
                    'too small a value slows progress.',
            'must_include': ['hyperparameter', 'update'],
            'why': 'abbreviation alias'},
           {'group': 'made_up',
            'question': 'What is the AIE_REVIEW_MODE setting?',
            'term': 'AIE_REVIEW_MODE',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': 'Looks like the real AIE_MODEL variable.'},
           {'group': 'made_up',
            'question': 'What is a lecNN-final tag?',
            'term': 'lecNN-final tag',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': 'Mirrors the real lecNN-start tag.'},
           {'group': 'made_up',
            'question': 'What is the Glossary Sync Agent?',
            'term': 'Glossary Sync Agent',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Contains the alias 'Agent' (search trap)."},
           {'group': 'made_up',
            'question': 'Briefly explain grade.md.',
            'term': 'grade.md',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': 'Mirrors answer.md; uses a training template.'},
           {'group': 'made_up',
            'question': 'What is the Pi review buffer?',
            'term': 'Pi review buffer',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Contains the term 'Pi' (search trap)."},
           {'group': 'made_up',
            'question': 'What is a harness ledger?',
            'term': 'harness ledger',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Contains the term 'Harness' (search trap)."},
           {'group': 'made_up',
            'question': 'What is tool fingerprinting?',
            'term': 'tool fingerprinting',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Contains the alias 'tool' (search trap)."},
           {'group': 'made_up',
            'question': 'What is the Lecture 04 capstone checkpoint?',
            'term': 'Lecture 04 capstone checkpoint',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': 'Course-structure style, no glossary word.'},
           {'group': 'made_up',
            'question': 'What is AIE_FALLBACK_MODEL?',
            'term': 'AIE_FALLBACK_MODEL',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': 'Environment-variable style, like AIE_MODEL.'},
           {'group': 'made_up',
            'question': 'What is a Codespaces snapshot policy?',
            'term': 'Codespaces snapshot policy',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Contains the term 'Codespaces' (search trap)."}]}

# Validation split: one held-out template for every term
VAL_TEMPLATE = "Define {term}."
val_records = [qa(VAL_TEMPLATE.format(term=t["term"]), t["definition"]) for t in TERMS]
V3_ITEMS = EVAL_V3["items"]
eval_q = {it["question"] for it in EVAL_ITEMS} | {it["question"] for it in V3_ITEMS}
val_q = {r["messages"][0]["content"] for r in val_records}
all_train_q = {r["messages"][0]["content"] for r in plain_records + alias_records + abstain_records}
assert not val_q & eval_q, f"Validation questions leak into evaluation: {val_q & eval_q}"
assert not val_q & all_train_q, "Validation questions overlap training"
assert not {it["question"] for it in V3_ITEMS} & all_train_q, "v3 evaluation questions found in training data"
assert not {it["question"] for it in V3_ITEMS} & {it["question"] for it in EVAL_ITEMS}, "v3 must not repeat the original 35"
for it in V3_ITEMS:
    if it["group"] == "made_up":
        assert norm(it["term"]) not in GLOSSARY_KEYS, it["term"]
    else:
        assert it["term_id"] in {t["id"] for t in TERMS}
print(f"val={len(val_records)} | v3 eval: alias={sum(i['group']=='alias' for i in V3_ITEMS)} made_up={sum(i['group']=='made_up' for i in V3_ITEMS)} | leakage checks OK")
(OUT / "eval_set_v3.json").write_text(json.dumps(EVAL_V3, ensure_ascii=False, indent=2), encoding="utf-8")

# ---------------------------------------------------------------- 3a. model and generation (v2 code wrapped in load_base)
if DRY_RUN:
    tok, DEVICE = None, "cpu"
    def set_seed(seed): pass
    def load_base(): return "stub-model"
    def generate_chat(model, messages):
        content = messages[-1]["content"]
        if "Definition: " in content: return content.split("Definition: ")[-1].split("\n\nQuestion")[0]
        if "glossary" in content.lower() or model == "stub-abstainer": return ABSTAIN_ANSWER
        return "A made-up definition."
else:
    import torch, gc
    from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
    if tok.pad_token is None: tok.pad_token = tok.eos_token

    def load_base():
        """Fresh 32-bit base model (same loading as v2)"""
        m = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
        m.config.use_cache = True
        log("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| weight memory (GB):", round(m.get_memory_footprint() / 1e9, 2))
        return m

    def generate_chat(model, messages: list):
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tok(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def build_messages(question: str):
    return [{"role": "user", "content": question}]

def generate(model, question: str):
    return generate_chat(model, build_messages(question))

def write_versions():
    """versions.json: exact environment of this run"""
    import platform, importlib.metadata as md
    def _v(pkg):
        try: return md.version(pkg)
        except md.PackageNotFoundError: return None
    versions = {"python": sys.version.split()[0], "platform": platform.platform(),
                **{p: _v(p) for p in ["torch", "transformers", "peft", "trl", "bitsandbytes", "datasets", "accelerate"]},
                "model": CONFIG["base_model"], "model_revision": CONFIG["revision"], "gpu": None, "cuda": None, "dry_run": DRY_RUN}
    if not DRY_RUN:
        versions["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
        versions["cuda"] = torch.version.cuda
    (OUT / "versions.json").write_text(json.dumps(versions, indent=2), encoding="utf-8")
    log("versions:", json.dumps(versions))

# ---------------------------------------------------------------- 3a. metrics (v2 cell, unchanged)
from collections import Counter
import pandas as pd
pd.set_option("display.max_colwidth", None)   # Display original outputs without truncation

GROUPS = ["in_glossary", "paraphrased", "absent"]
# "Not in the glossary / I don't know" phrasings counted as abstention
ABSTAIN_PATTERNS = [
    r"\bnot (?:in|included in|listed in|found in|part of|covered (?:in|by)|defined in|present in|mentioned in) (?:the |this |our )?(?:course(?:'s)? )?glossary",
    r"\bglossary (?:does not|doesn't|did not) (?:contain|include|list|cover|define|mention)",
    r"\b(?:no|not an?) (?:entry|definition|information) (?:for|on|about|in)\b",
    r"\b(?:i )?(?:do not|don't|cannot|can't) (?:know|find|answer)\b",
    r"\b(?:not sure|not familiar|unable to (?:find|answer)|no information)\b",
]

def is_abstention(text: str) -> bool:
    return any(re.search(p, text.lower()) for p in ABSTAIN_PATTERNS)

def tokens(text: str) -> list:
    return re.findall(r"[0-9a-z]+", text.lower())

def token_f1(pred: str, gold: str) -> float:
    """SQuAD-style token F1 (bag of lowercase alphanumeric tokens)"""
    p, g = tokens(pred), tokens(gold)
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0: return 0.0
    precision, recall = common / len(p), common / len(g)
    return 2 * precision * recall / (precision + recall)

def has_keyword(text: str, kw: str) -> bool:
    """Case-insensitive whole-word match (so 'PR' would not match 'pretrained')"""
    return re.search(r"(?<!\w)" + re.escape(kw.lower()) + r"(?!\w)", text.lower()) is not None

def score(item: dict, output: str) -> dict:
    abstained = is_abstention(output)
    row = {"group": item["group"], "question": item["question"], "expected": item.get("term_id") or "(absent)", "output": output, "abstained": abstained}
    if item["group"] == "absent":
        row.update(keyword_hit=None, keyword_frac=None, f1=None, verdict="abstained" if abstained else "asserted", ok=abstained)
    else:
        hits = [has_keyword(output, k) for k in item["must_include"]]
        row.update(keyword_hit=all(hits), keyword_frac=sum(hits) / len(hits), f1=round(token_f1(output, item["gold"]), 3),
                   verdict="abstained" if abstained else ("hit" if all(hits) else "partial" if any(hits) else "miss"), ok=all(hits))
    return row

def summarize(rows: list, label: str, seconds: float) -> dict:
    """Per-group summary. Glossary groups: keyword hit rate, keyword fraction, F1, false-abstention rate. Absent: abstain vs assert rate"""
    out = {"label": label, "seconds": round(seconds, 1)}
    for g in GROUPS:
        rs = [r for r in rows if r["group"] == g]
        s = {"n": len(rs), "abstain_rate": round(sum(r["abstained"] for r in rs) / len(rs), 3)}
        if g == "absent":
            s["assert_rate"] = round(1 - s["abstain_rate"], 3)
        else:
            s.update(keyword_hit_rate=round(sum(r["keyword_hit"] for r in rs) / len(rs), 3),
                     keyword_frac=round(sum(r["keyword_frac"] for r in rs) / len(rs), 3),
                     f1=round(sum(r["f1"] for r in rs) / len(rs), 3))
        out[g] = s
    return out

def summary_table(*summaries) -> pd.DataFrame:
    """One row per (label, group) for side-by-side display"""
    return pd.DataFrame([{"label": s["label"], "group": g, **s[g]} for s in summaries for g in GROUPS]).set_index(["group", "label"]).sort_index(level=0, sort_remaining=False)

# ---------------------------------------------------------------- 5. glossary search and search + model (v2 cells, unchanged)
import difflib

FUZZY_THRESHOLD = 0.88
STOPWORDS = set("a an the is are was what which who how why when where do does did i my me we our you your of in on for to and or with by from this that it its be can about mean means explain briefly tell".split())

# Index: (normalized key, original spelling, term) for every term, term part, and alias
INDEX = [(norm(k), k, t) for t in TERMS for k in glossary_keys(t) if len(norm(k)) > 1]

def glossary_search(question: str):
    """Return (answer, matched term id or None, how it matched)"""
    q = f" {norm(question)} "
    exact = [(len(key) + (3 if any(ch.isupper() for ch in orig) and re.search(rf"(?<!\w){re.escape(orig)}(?!\w)", question) else 0), key, t)
             for key, orig, t in INDEX if f" {key} " in q]
    if exact:
        _, key, t = max(exact, key=lambda x: x[0])
        return t["definition"], t["id"], f"exact:{key}"
    words = q.split()
    spans = {" ".join(words[i:j]) for i in range(len(words)) for j in range(i + 1, min(i + 4, len(words)) + 1)}
    spans = [s for s in spans if len(s) >= 4 and not set(s.split()) <= STOPWORDS]
    best = max(((difflib.SequenceMatcher(None, s, key).ratio(), s, key, t) for s in spans for key, _, t in INDEX if len(key) >= 4),
               key=lambda x: x[0], default=None)
    if best and best[0] >= FUZZY_THRESHOLD:
        ratio, span, key, t = best
        return t["definition"], t["id"], f"fuzzy:{span}~{key}({ratio:.2f})"
    return ABSTAIN_ANSWER, None, "none"

def evaluate_search(label: str = "search"):
    rows, latencies = [], []
    for item in EVAL_ITEMS:
        t0 = time.perf_counter()
        answer, term_id, how = glossary_search(item["question"])
        latencies.append(time.perf_counter() - t0)
        rows.append({**score(item, answer), "retrieved": term_id, "match": how,
                     "retrieved_correct": term_id == item.get("term_id")})   # absent: correct only if nothing was retrieved
    summary = summarize(rows, label, sum(latencies))
    summary["avg_latency_ms"] = round(1000 * sum(latencies) / len(latencies), 3)
    summary["retrieval_accuracy"] = {g: round(sum(r["retrieved_correct"] for r in rows if r["group"] == g) / sum(r["group"] == g for r in rows), 3) for g in GROUPS}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    return summary, rows

RAG_INSTRUCTION = ("Answer the question using only the course glossary entry below. "
                   "Do not add information that is not in the entry.")

def build_rag_messages(question: str, entry: dict) -> list:
    """One user message: instruction + retrieved glossary entry (term and definition) + question"""
    return [{"role": "user", "content": f"{RAG_INSTRUCTION}\n\nGlossary entry:\nTerm: {entry['term']}\nDefinition: {entry['definition']}\n\nQuestion: {question}"}]

TERM_BY_ID = {t["id"]: t for t in TERMS}

def rag_answer(question: str, gen_fn):
    """gen_fn(messages) -> str. Returns (answer, retrieved id or None, how matched, whether the model was called)"""
    _, term_id, how = glossary_search(question)
    if term_id is None:
        return ABSTAIN_ANSWER, None, how, False          # Fixed abstention, no model call
    return gen_fn(build_rag_messages(question, TERM_BY_ID[term_id])), term_id, how, True

# v3: the v2 metric functions above are unchanged; these wrappers let them score the v3 groups too.
# alias is scored like paraphrased (keywords + F1), made_up like absent (abstain vs assert).
ABSENT_LIKE = {"absent", "made_up"}
GROUPS_V1 = ["in_glossary", "paraphrased", "absent"]
GROUPS_V3 = ["alias", "made_up"]

def score_any(item, output):
    row = score({**item, "group": "absent" if item["group"] in ABSENT_LIKE else "paraphrased"}, output)
    row["group"] = item["group"]
    return row

def summarize_any(rows, label, seconds, groups):
    out = {"label": label, "seconds": round(seconds, 1)}
    for g in groups:
        rs = [r for r in rows if r["group"] == g]
        s = {"n": len(rs), "abstain_rate": round(sum(r["abstained"] for r in rs) / len(rs), 3)}
        if g in ABSENT_LIKE:
            s["assert_rate"] = round(1 - s["abstain_rate"], 3)
        else:
            s.update(keyword_hit_rate=round(sum(r["keyword_hit"] for r in rs) / len(rs), 3),
                     keyword_frac=round(sum(r["keyword_frac"] for r in rs) / len(rs), 3),
                     f1=round(sum(r["f1"] for r in rs) / len(rs), 3))
        out[g] = s
    return out

SETS = {"v1": (EVAL_ITEMS, GROUPS_V1), "v3": (V3_ITEMS, GROUPS_V3)}

def save_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def run_model_eval(model, label, set_name, folder):
    items, groups = SETS[set_name]
    rows, t0 = [], time.time()
    for i, it in enumerate(items, 1):
        rows.append(score_any(it, generate(model, it["question"])))
        if i % 10 == 0: log(f"  {label}/{set_name}: {i}/{len(items)}")
    summary = summarize_any(rows, label, time.time() - t0, groups)
    save_json(folder / f"eval_{label}_{set_name}.json", {"summary": summary, "rows": rows})
    return summary, rows

def run_search_eval(set_name, folder):
    items, groups = SETS[set_name]
    rows, lat = [], []
    for it in items:
        t0 = time.perf_counter(); answer, term_id, how = glossary_search(it["question"]); lat.append(time.perf_counter() - t0)
        rows.append({**score_any(it, answer), "retrieved": term_id, "match": how, "retrieved_correct": term_id == it.get("term_id")})
    summary = summarize_any(rows, "search", sum(lat), groups)
    summary["avg_latency_ms"] = round(1000 * sum(lat) / len(lat), 3)
    save_json(folder / f"eval_search_{set_name}.json", {"summary": summary, "rows": rows})
    return summary, rows

def run_rag_eval(model, label, set_name, folder):
    items, groups = SETS[set_name]
    rows, t0 = [], time.time()
    for i, it in enumerate(items, 1):
        if i % 10 == 0: log(f"  {label}/{set_name}: {i}/{len(items)}")
        answer, term_id, how, called = rag_answer(it["question"], lambda m: generate_chat(model, m))
        rows.append({**score_any(it, answer), "retrieved": term_id, "match": how, "model_called": called,
                     "retrieved_correct": term_id == it.get("term_id")})
    summary = summarize_any(rows, label, time.time() - t0, groups)
    summary["model_calls"] = sum(r["model_called"] for r in rows)
    save_json(folder / f"eval_{label}_{set_name}.json", {"summary": summary, "rows": rows})
    return summary, rows

# ---------------------------------------------------------------- 3. seed-independent runs: before, search, search + base
SHARED = OUT / "shared"
SHARED_PLAN = [(label, s) for s in SETS for label in ("before", "search", "rag_base")]

def load_eval(path):
    d = json.loads(path.read_text(encoding="utf-8"))
    return d["summary"], d["rows"]

def shared_done():
    return all((SHARED / f"eval_{l}_{s}.json").exists() for l, s in SHARED_PLAN)

def stage_shared():
    write_versions()
    todo = [(l, s) for l, s in SHARED_PLAN if not (SHARED / f"eval_{l}_{s}.json").exists()]
    model = load_base() if any(l != "search" for l, _ in todo) else None
    for label, s in todo:
        t0 = time.time()
        if label == "before": summ, _ = run_model_eval(model, "before", s, SHARED)
        elif label == "search": summ, _ = run_search_eval(s, SHARED)
        else: summ, _ = run_rag_eval(model, "rag_base", s, SHARED)
        log(f"shared {label}/{s} done in {time.time() - t0:.0f} s:", json.dumps({g: summ[g] for g in SETS[s][1]}))

# ---------------------------------------------------------------- 4. one seed: train with validation loss, evaluate
def stage_seed(seed):
    folder = OUT / f"seed_{seed}"
    if (folder / "done.json").exists():
        log(f"seed {seed}: already done, skipped"); return
    train_ds_rows, val_ds_rows = train_records, val_records
    if DRY_RUN:
        time.sleep(float(os.environ.get("LEC02_DRY_SLEEP", "0")))
        set_seed(seed)
        trainable, total, val_before, train_seconds, global_step = 1, 100, 2.0, 0.0, 20
        log_hist = [{"step": 10, "epoch": 0.5, "loss": 1.0}, {"step": 20, "epoch": 1.0, "eval_loss": 0.9}, {"step": 20, "epoch": 1.0, "loss": 0.2}]
        tuned = "stub-abstainer" if seed == SEEDS[0] else "stub-model"
    else:
        from peft import LoraConfig
        from trl import SFTTrainer, SFTConfig
        from datasets import Dataset
        model = load_base(); model.config.use_cache = False
        set_seed(seed)   # before the adapter is created, so the adapter initialisation depends on the seed
        lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                          target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
        sft = SFTConfig(output_dir=str(WORK / f"ckpt_{RUN_NAME}_{seed}"), num_train_epochs=CONFIG["num_train_epochs"],
                        per_device_train_batch_size=4, gradient_accumulation_steps=2, per_device_eval_batch_size=8,
                        learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"],
                        logging_steps=10, eval_strategy="epoch", save_strategy="no", fp16=False,
                        max_length=CONFIG["max_seq_len"], packing=False, seed=seed,
                        report_to="none", disable_tqdm=True)   # no progress bars / widgets: Trainer prints log dicts instead
        trainer = SFTTrainer(model=model, args=sft, train_dataset=Dataset.from_list(train_ds_rows),
                             eval_dataset=Dataset.from_list(val_ds_rows), peft_config=lora, processing_class=tok,
                             callbacks=[heartbeat_callback()])
        trainable, total = trainer.model.get_nb_trainable_parameters()
        log(f"seed {seed} | trainable {trainable:,} / {total:,} ({100 * trainable / total:.3f}%)")
        val_before = trainer.evaluate()["eval_loss"]   # validation loss at step 0 (adapter initialised to a no-op)
        log(f"seed {seed} | validation loss before training {val_before:.4f}")
        t0 = time.time(); trainer.train(); train_seconds = time.time() - t0
        log_hist, global_step = trainer.state.log_history, trainer.state.global_step
        tuned = trainer.model; tuned.config.use_cache = True; tuned.eval()
    loss = {"seed": seed, "run_name": RUN_NAME, "trainable_params": trainable, "total_params": total,
            "train_seconds": round(train_seconds, 1), "global_step": global_step,
            "train_loss": [{"step": e["step"], "epoch": e["epoch"], "loss": e["loss"]} for e in log_hist if "loss" in e],
            "val_loss": [{"step": 0, "epoch": 0.0, "eval_loss": val_before}] +
                        [{"step": e["step"], "epoch": e["epoch"], "eval_loss": e["eval_loss"]} for e in log_hist if "eval_loss" in e],
            "log_history": log_hist}
    save_json(folder / "loss.json", loss)
    log(f"seed {seed} | trained in {train_seconds:.0f} s, {global_step} steps; loss.json saved")
    res = {"seed": seed, "run_name": RUN_NAME, "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir", "seeds")},
           "trainable_params": trainable, "train_seconds": loss["train_seconds"], "train_steps": global_step,
           "final_train_loss": loss["train_loss"][-1]["loss"] if loss["train_loss"] else None,
           "final_val_loss": loss["val_loss"][-1]["eval_loss"], "min_val_loss": min(v["eval_loss"] for v in loss["val_loss"]),
           "train_set_size": len(train_records), "val_set_size": len(val_records), "sft_jsonl_sha256": SFT_SHA256}
    for s in SETS:
        a_sum, a_rows = run_model_eval(tuned, "after", s, folder)
        r_sum, _ = run_rag_eval(tuned, "rag_trained", s, folder)
        b_rows = load_eval(SHARED / f"eval_before_{s}.json")[1]
        res[s] = {"after": a_sum, "rag_trained": r_sum,
                  "regressed": [b["question"] for b, a in zip(b_rows, a_rows) if b["ok"] and not a["ok"]],
                  "improved": [b["question"] for b, a in zip(b_rows, a_rows) if not b["ok"] and a["ok"]]}
        log(f"seed {seed} | {s} after:", json.dumps({g: a_sum[g] for g in SETS[s][1]}))
    save_json(folder / "result.json", res)
    save_json(folder / "done.json", {"seed": seed, "finished": time.strftime("%Y-%m-%d %H:%M:%S")})   # written last
    log(f"seed {seed} | done")

# ---------------------------------------------------------------- 7. summary over seeds and loss curves
def stage_summary():
    import statistics
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import pandas as pd
    done = [s for s in SEEDS if (OUT / f"seed_{s}" / "done.json").exists()]
    log("finished seeds:", done, "| missing:", [s for s in SEEDS if s not in done])
    per_seed = {s: json.loads((OUT / f"seed_{s}" / "result.json").read_text(encoding="utf-8")) for s in done}
    losses = {s: json.loads((OUT / f"seed_{s}" / "loss.json").read_text(encoding="utf-8")) for s in done}
    shared = {(l, s): load_eval(SHARED / f"eval_{l}_{s}.json") for l, s in SHARED_PLAN}
    versions = json.loads((OUT / "versions.json").read_text(encoding="utf-8")) if (OUT / "versions.json").exists() else None

    def agg(values):
        values = [v for v in values if v is not None]
        if not values: return None
        return {"mean": round(statistics.mean(values), 4), "min": min(values), "max": max(values), "n": len(values), "values": values}

    summary = {"run_name": RUN_NAME, "seeds": done, "versions": versions, "conditions": {}}
    for s, (_, groups) in SETS.items():
        for cond in ("before", "search", "rag_base"):
            sm = shared[(cond, s)][0]
            summary["conditions"].setdefault(cond, {})[s] = {g: sm[g] for g in groups}
        for cond in ("after", "rag_trained"):
            summary["conditions"].setdefault(cond, {})[s] = {
                g: {m: agg([per_seed[sd][s][cond][g].get(m) for sd in done]) for m in per_seed[done[0]][s][cond][g] if m != "n"}
                for g in groups} if done else None
        summary.setdefault("regressed", {})[s] = {sd: per_seed[sd][s]["regressed"] for sd in done}
    summary["training"] = {k: agg([per_seed[sd][k] for sd in done])
                           for k in ("train_seconds", "final_train_loss", "final_val_loss", "min_val_loss", "trainable_params")}
    save_json(OUT / f"summary_{RUN_NAME}.json", summary)
    log("training:", json.dumps(summary["training"]))

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for i, s in enumerate(done):
        c = f"C{i}"
        tr = losses[s]["train_loss"]; va = losses[s]["val_loss"]
        ax.plot([e["epoch"] for e in tr], [e["loss"] for e in tr], color=c, lw=1, label=f"train, seed {s}")
        ax.plot([e["epoch"] for e in va], [e["eval_loss"] for e in va], color=c, lw=2, ls="--", marker="o", ms=3, label=f"validation, seed {s}")
    ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_yscale("log"); ax.grid(alpha=.3)
    ax.set_title(f"{RUN_NAME}: training vs validation loss ('Define {{term}}.' template)"); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / f"loss_curves_v3_{RUN_NAME}.png", dpi=150); plt.close(fig)

    rows = []   # primary metric per group: F1 for alias, keyword hit for in_glossary/paraphrased, abstain rate for absent/made_up
    for s, (_, groups) in SETS.items():
        for cond, d in summary["conditions"].items():
            for g in groups:
                v = d[s][g] if d.get(s) else None
                if not v: continue
                key = "abstain_rate" if g in ABSENT_LIKE else ("f1" if g == "alias" else "keyword_hit_rate")
                x = v[key]
                rows.append({"set": s, "group": g, "condition": cond, "metric": key, "value": x["mean"] if isinstance(x, dict) else x,
                             "min-max": f"{x['min']}-{x['max']}" if isinstance(x, dict) else "-"})
    table = pd.DataFrame(rows)
    table.to_csv(OUT / f"summary_table_{RUN_NAME}.csv", index=False)
    display(table)

# ---------------------------------------------------------------- orchestration (the detached process)
def orchestrate():
    lock = OUT / "run.lock"
    acquire_lock(lock)
    try:
        (OUT / "ALL_DONE.json").unlink(missing_ok=True)
        log(f"run {RUN_NAME} | seeds {SEEDS} | out {OUT} | pid {os.getpid()}{' | DRY RUN' if DRY_RUN else ''}")
        require_gpu()
        if not shared_done() and not run_stage(["--stage", "shared"]):
            log("shared stage failed; stopping (rerun LAUNCH to retry)"); return 1
        failed = []
        for seed in SEEDS:
            if (OUT / f"seed_{seed}" / "done.json").exists():
                log(f"seed {seed}: already done, skipped"); continue
            if not run_stage(["--stage", "seed", "--seed", str(seed)]):
                failed.append(seed)
        run_stage(["--stage", "summary"], retries=0)
        status = {"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "seeds": SEEDS,
                  "seeds_done": [s for s in SEEDS if (OUT / f"seed_{s}" / "done.json").exists()], "seeds_failed": failed}
        save_json(OUT / "STATUS.json", status)
        if not failed:
            save_json(OUT / "ALL_DONE.json", status); log("ALL DONE")
        else:
            log("finished with failed seeds:", failed, "- rerun LAUNCH to retry them")
        return 0 if not failed else 1
    finally:
        release_lock(lock)

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", choices=["all", "shared", "seed", "summary"], default="all")
    ap.add_argument("--seed", type=int)
    a = ap.parse_args()
    if a.stage == "all": sys.exit(orchestrate())
    elif a.stage == "shared": stage_shared()
    elif a.stage == "seed": stage_seed(a.seed)
    else: stage_summary()

In [ ]:
# [Cell 5] LAUNCH: start the run in the background and return at once (safe to rerun: never starts a second copy)
if (OUT / "ALL_DONE.json").exists():
    print("Already finished. Go to the FINISH cell.")
elif run_state():
    print(f"Already running (pid {run_state()['pid']}, started {run_state()['started']}). Use the POLL cell; do not launch again.")
else:
    shutil.copy(RUN_PY, OUT / RUN_PY.name)   # keep the exact script next to the results
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(f"\n===== launch {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
    cmd = f"nohup {shlex.quote(sys.executable)} -u {shlex.quote(str(RUN_PY))} --stage all >> {shlex.quote(str(LOG))} 2>&1 &"
    subprocess.Popen(cmd, shell=True, env={**os.environ, "LEC02_OUT": str(OUT)}, start_new_session=True,
                     stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(LAUNCH_WAIT_S)
    print("running:", bool(run_state()), "| log:", LOG)
    print(tail(10))

## Poll

Rerun this cell every 10–20 minutes. Training lines look like `step 35/300 | epoch 2.33 | 140 s`.

In [ ]:
# [Cell 6] POLL: rerun whenever you like; prints status and the last 20 log lines, then exits
state = run_state()
age = time.time() - LOG.stat().st_mtime if LOG.exists() else None
age_txt = f"last log write {age:.0f} s ago" if age is not None else "no log yet"
print(f"{time.strftime('%H:%M:%S')} | running: {bool(state)}{' (pid ' + str(state['pid']) + ')' if state else ''}"
      f" | finished: {(OUT / 'ALL_DONE.json').exists()} | {age_txt}")
print("done markers:", sorted(str(p.parent.relative_to(OUT)) for p in OUT.glob("*/done.json")))
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip())
except Exception:
    pass
print("-" * 80); print(tail(20))
if not state and not (OUT / "ALL_DONE.json").exists() and LOG.exists():
    print("\nNOT running and not finished: rerun the LAUNCH cell to resume (finished parts are skipped).")

## Finish

When POLL shows `finished: True` (or `ALL DONE` in the log): summary, loss curves, zip.

In [ ]:
# [Cell 7] FINISH: check completion, rebuild the summary / plots, zip the results
WAIT_MINUTES = 0   # >0 waits here (one short line per minute) until the run finishes; 0 = just check now
t_end = time.time() + 60 * WAIT_MINUTES
while not (OUT / "ALL_DONE.json").exists() and run_state() and time.time() < t_end:
    time.sleep(60); print(time.strftime("%H:%M:%S"), "still running |", tail(1))
if not (OUT / "ALL_DONE.json").exists():
    print("Not finished yet:", "still running - use POLL and come back" if run_state() else
          "stopped early - rerun LAUNCH to resume (see STATUS.json / log.txt)")
else:
    r = subprocess.run([sys.executable, "-u", str(RUN_PY), "--stage", "summary"], env={**os.environ, "LEC02_OUT": str(OUT)},
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-2000:] if r.returncode else "")
    summary = json.loads((OUT / f"summary_{RUN_NAME}.json").read_text(encoding="utf-8"))
    print("seeds:", summary["seeds"]); print(json.dumps(summary["training"], indent=1)[:1500])
    try:
        from IPython.display import Image, display
        display(Image(filename=str(OUT / f"loss_curves_v3_{RUN_NAME}.png")))
    except Exception as e:
        print("plot saved at", OUT / f"loss_curves_v3_{RUN_NAME}.png", f"({type(e).__name__})")
    archive = shutil.make_archive(str(LOCAL / f"lec02_v3_{RUN_NAME}"), "zip", root_dir=OUT)
    if Path(archive).resolve() != (OUT.parent / Path(archive).name).resolve():
        shutil.copy(archive, OUT.parent / Path(archive).name)
    print("zip:", archive, "| copy on Drive:", OUT.parent / Path(archive).name)
    try:
        from google.colab import files
        files.download(archive)
    except Exception as e:
        print(f"Automatic download unavailable ({type(e).__name__}); download the zip from the Files panel or Drive")